<a href="https://colab.research.google.com/github/PamellaElis/AlgorithmicBias/blob/main/SLR_Bias.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ===========================
# EN STEP 1 - GOOGLE DRIVE CONFIGURATION
# PT ETAPA 1 - CONFIGURAR O GOOGLE DRIVE
# ===========================

import os
from google.colab import drive

MOUNT_POINT = "/content/drive"
DATA_DIR = "/content/drive/MyDrive/Tese/ArticleSLR"

# ---------------------------
# 1️⃣ Safely unmount if already mounted
# ---------------------------
if os.path.ismount(MOUNT_POINT):
    print("🔄 Drive already mounted. Unmounting...")
    drive.flush_and_unmount()

# Remove mount directory if it contains files (prevents ValueError)
if os.path.exists(MOUNT_POINT) and os.listdir(MOUNT_POINT):
    print("🧹 Cleaning previous mount directory...")
    !rm -rf /content/drive

# ---------------------------
# 2️⃣ Mount Google Drive
# ---------------------------
print("📌 Mounting Google Drive...")
drive.mount(MOUNT_POINT)

# ---------------------------
# 3️⃣ Ensure project directory exists
# ---------------------------
os.makedirs(DATA_DIR, exist_ok=True)

# ---------------------------
# 4️⃣ Set working directory
# ---------------------------
os.chdir(DATA_DIR)

print("\n📁 Current working directory:")
print(os.getcwd())

print("\n📄 Available files:")
files = os.listdir(".")
if files:
    for f in files:
        print(" -", f)
else:
    print("⚠️ No files found in this directory.")


📌 Mounting Google Drive...
Mounted at /content/drive

📁 Current working directory:
/content/drive/MyDrive/Tese/ArticleSLR

📄 Available files:
 - WebOfScience_Data
 - JCR_Data
 - Scopus_Data
 - ArXiv_Data
 - scopus.clean.csv
 - wos.clean.csv
 - arxiv.clean.csv
 - data1.unification.csv
 - data0.duplicated.csv
 - data2.unique.csv
 - data0.journals.csv
 - jcr1.clean.csv
 - jcr2.unique.csv
 - data3.impact.csv
 - data4.q1q2.csv
 - data5.pillar_included.csv
 - data0.pillar_excluded.csv
 - PDFs
 - api_cache.json
 - data6.checkpoint.csv
 - data6.pdf.csv
 - data0.nopdf.csv
 - PDFs_Zotero
 - data8.pdf.csv
 - data0.nopdf2.csv
 - data7.pdf.csv
 - data0.nopdf2.bib
 - data9.pdf.csv
 - data0.nopdf3.csv
 - data0.Q1_24a26_BusinessManagement.csv
 - data0.nopdf4 (21 faltantes).gsheet
 - data10.pdf.csv
 - data0.nopdf4.csv
 - data11.pdf.csv
 - data0.nopdf5.csv
 - artigos_analise.csv
 - artigos_analise.xlsx
 - log_erros.csv
 - artigos_analise.gsheet
 - PDFs_Q1_24a26_BusinessManagement
 - PDFs_manually
 - PDF

In [ ]:
# ==========================================================
# EN — STEP 2: EXTRACT AND STANDARDIZE SCOPUS DATA
# PT — ETAPA 2: EXTRAIR E PADRONIZAR DADOS DO SCOPUS
# ==========================================================

import pandas as pd
import os
import glob
from pathlib import Path

# ----------------------------------------------------------
# EN — 1️⃣ DEFINE DIRECTORIES AND OUTPUT FILE
# PT — 1️⃣ DEFINIR DIRETÓRIOS E ARQUIVO DE SAÍDA
# ----------------------------------------------------------

SCOPUS_DIR = Path("/content/drive/MyDrive/Tese/ArticleSLR/Scopus_Data")
DATA_DIR = Path("/content/drive/MyDrive/Tese/ArticleSLR")
OUTPUT_FILENAME = "scopus.clean.csv"
OUTPUT_PATH = DATA_DIR / OUTPUT_FILENAME

# ----------------------------------------------------------
# EN — 2️⃣ VALIDATE INPUT DIRECTORY
# PT — 2️⃣ VALIDAR DIRETÓRIO DE ENTRADA
# ----------------------------------------------------------

if not SCOPUS_DIR.exists():
    raise FileNotFoundError(
        f"❌ EN — Directory not found: {SCOPUS_DIR}\n"
        f"❌ PT — Diretório não encontrado: {SCOPUS_DIR}"
    )

# ----------------------------------------------------------
# EN — 3️⃣ LOAD ALL SCOPUS CSV FILES
# PT — 3️⃣ CARREGAR TODOS OS ARQUIVOS CSV DO SCOPUS
# ----------------------------------------------------------

csv_files = sorted(glob.glob(str(SCOPUS_DIR / "*.csv")))

if not csv_files:
    raise FileNotFoundError(
        f"❌ EN — No .csv files found in: {SCOPUS_DIR}\n"
        f"❌ PT — Nenhum arquivo .csv encontrado em: {SCOPUS_DIR}"
    )

print(f"📂 EN — {len(csv_files)} CSV file(s) found.")
print(f"📂 PT — {len(csv_files)} arquivo(s) CSV encontrado(s).\n")

df_list = []

for file_path in csv_files:
    print(f"📄 EN — Loading: {Path(file_path).name}")
    print(f"📄 PT — Carregando: {Path(file_path).name}")

    df = pd.read_csv(file_path)

    # EN — Normalize column names (lowercase + strip spaces)
    # PT — Normalizar nomes das colunas (minúsculo + remover espaços)
    df.columns = df.columns.str.strip().str.lower()

    df_list.append(df)

# ----------------------------------------------------------
# EN — 4️⃣ CONCATENATE ALL DATAFRAMES
# PT — 4️⃣ CONCATENAR TODOS OS DATAFRAMES
# ----------------------------------------------------------

df_all = pd.concat(df_list, ignore_index=True)

print("\n📊 EN — Combined dataset shape:")
print("📊 PT — Dimensão do dataset combinado:")
print(df_all.shape)

# ----------------------------------------------------------
# EN — 5️⃣ STANDARD COLUMN MAPPING
# PT — 5️⃣ MAPEAMENTO PADRÃO DE COLUNAS
# ----------------------------------------------------------

column_map = {
    "Title": ["title"],
    "Authors": ["authors"],
    "Year": ["year", "publication"],
    "DOI": ["doi"],
    "Abstract": ["abstract"],
    "Journal Name": ["source title", "journal name"],
    "Journal Area": ["research areas", "web of science categories", "category", "subject area"],
    "ISSN Code": ["issn", "issn code"],
    "eISSN Code": ["eissn", "eissn code"],
    "Document Type": ["document type", "doctype"]
}

# ----------------------------------------------------------
# EN — 6️⃣ BUILD STANDARDIZED DATAFRAME
# PT — 6️⃣ CONSTRUIR DATAFRAME PADRONIZADO
# ----------------------------------------------------------

filtered_data = {}

for standard_col, possible_names in column_map.items():

    matched_col = next((col for col in possible_names if col in df_all.columns), None)

    if matched_col:
        filtered_data[standard_col] = df_all[matched_col]
    else:
        # EN — If column not found, fill with placeholder
        # PT — Se coluna não encontrada, preencher com placeholder
        filtered_data[standard_col] = ["-"] * len(df_all)

scopus_filtered = pd.DataFrame(filtered_data)

# EN — Replace missing values with "-"
# PT — Substituir valores ausentes por "-"
scopus_filtered.fillna("-", inplace=True)

# ----------------------------------------------------------
# EN — 7️⃣ SAVE CLEANED DATASET
# PT — 7️⃣ SALVAR DATASET LIMPO
# ----------------------------------------------------------

scopus_filtered.to_csv(OUTPUT_PATH, index=False, encoding="utf-8")

# ----------------------------------------------------------
# EN — 8️⃣ SUMMARY REPORT
# PT — 8️⃣ RELATÓRIO RESUMIDO
# ----------------------------------------------------------

print("\n✅ EN — Clean file saved successfully.")
print("✅ PT — Arquivo limpo salvo com sucesso.")

print(f"\n📍 EN — Output path: {OUTPUT_PATH}")
print(f"📍 PT — Caminho de saída: {OUTPUT_PATH}")

print(f"\n📊 EN — Final shape: {scopus_filtered.shape[0]} rows × {scopus_filtered.shape[1]} columns")
print(f"📊 PT — Dimensão final: {scopus_filtered.shape[0]} linhas × {scopus_filtered.shape[1]} colunas")

print("\n📑 EN — Final columns:")
print("📑 PT — Colunas finais:")
for col in scopus_filtered.columns:
    print(" -", col)


📂 EN — 1 CSV file(s) found.
📂 PT — 1 arquivo(s) CSV encontrado(s).

📄 EN — Loading: scopus_export_Feb 1-2026_2d98d217-bb6f-45c4-ab10-2fe4e4bc7ae3.csv
📄 PT — Carregando: scopus_export_Feb 1-2026_2d98d217-bb6f-45c4-ab10-2fe4e4bc7ae3.csv

📊 EN — Combined dataset shape:
📊 PT — Dimensão do dataset combinado:
(8380, 19)

✅ EN — Clean file saved successfully.
✅ PT — Arquivo limpo salvo com sucesso.

📍 EN — Output path: /content/drive/MyDrive/Tese/ArticleSLR/scopus.clean.csv
📍 PT — Caminho de saída: /content/drive/MyDrive/Tese/ArticleSLR/scopus.clean.csv

📊 EN — Final shape: 8380 rows × 10 columns
📊 PT — Dimensão final: 8380 linhas × 10 colunas

📑 EN — Final columns:
📑 PT — Colunas finais:
 - Title
 - Authors
 - Year
 - DOI
 - Abstract
 - Journal Name
 - Journal Area
 - ISSN Code
 - eISSN Code
 - Document Type


In [ ]:
# ==========================================================
# EN — STEP 3: EXTRACT AND STANDARDIZE WEB OF SCIENCE DATA
# PT — ETAPA 3: EXTRAIR E PADRONIZAR DADOS DA WEB OF SCIENCE
# ==========================================================

import pandas as pd
from pathlib import Path
from glob import glob

# ----------------------------------------------------------
# EN — 1️⃣ DEFINE DIRECTORIES AND OUTPUT FILE
# PT — 1️⃣ DEFINIR DIRETÓRIOS E ARQUIVO DE SAÍDA
# ----------------------------------------------------------

WOS_DIR = Path("/content/drive/MyDrive/Tese/ArticleSLR/WebOfScience_Data")
PROJECT_ROOT = Path("/content/drive/MyDrive/Tese/ArticleSLR")
OUTPUT_FILENAME = "wos.clean.csv"
OUTPUT_PATH = PROJECT_ROOT / OUTPUT_FILENAME

# ----------------------------------------------------------
# EN — 2️⃣ VALIDATE INPUT DIRECTORY
# PT — 2️⃣ VALIDAR DIRETÓRIO DE ENTRADA
# ----------------------------------------------------------

if not WOS_DIR.exists():
    raise FileNotFoundError(
        f"❌ EN — Directory not found: {WOS_DIR}\n"
        f"❌ PT — Diretório não encontrado: {WOS_DIR}"
    )

# ----------------------------------------------------------
# EN — 3️⃣ SEARCH FOR EXCEL FILES (.xls / .xlsx)
# PT — 3️⃣ BUSCAR ARQUIVOS EXCEL (.xls / .xlsx)
# ----------------------------------------------------------

wos_files = sorted(glob(str(WOS_DIR / "*.xls*")))

if not wos_files:
    raise FileNotFoundError(
        f"❌ EN — No Excel files found in: {WOS_DIR}\n"
        f"❌ PT — Nenhum arquivo Excel encontrado em: {WOS_DIR}"
    )

print(f"📂 EN — {len(wos_files)} Excel file(s) found.")
print(f"📂 PT — {len(wos_files)} arquivo(s) Excel encontrado(s).\n")

# ----------------------------------------------------------
# EN — 4️⃣ LOAD AND NORMALIZE EACH FILE
# PT — 4️⃣ CARREGAR E NORMALIZAR CADA ARQUIVO
# ----------------------------------------------------------

dfs = []

for file_path in wos_files:

    file_path = Path(file_path)

    print(f"📄 EN — Loading: {file_path.name}")
    print(f"📄 PT — Carregando: {file_path.name}")

    # EN — Select correct engine for .xlsx files
    # PT — Selecionar engine correta para arquivos .xlsx
    if file_path.suffix == ".xlsx":
        df = pd.read_excel(file_path, engine="openpyxl")
    else:
        df = pd.read_excel(file_path)

    # EN — Normalize column names (lowercase + strip spaces)
    # PT — Normalizar nomes das colunas (minúsculo + remover espaços)
    df.columns = df.columns.str.strip().str.lower()

    dfs.append(df)

# ----------------------------------------------------------
# EN — 5️⃣ CONCATENATE ALL DATAFRAMES
# PT — 5️⃣ CONCATENAR TODOS OS DATAFRAMES
# ----------------------------------------------------------

df_all = pd.concat(dfs, ignore_index=True)

print("\n📊 EN — Combined dataset shape:")
print("📊 PT — Dimensão do dataset combinado:")
print(df_all.shape)

# ----------------------------------------------------------
# EN — 6️⃣ DEFINE STANDARD COLUMN MAPPING
# PT — 6️⃣ DEFINIR MAPEAMENTO PADRÃO DE COLUNAS
# ----------------------------------------------------------

column_map = {
    "Title": ["title", "article title"],
    "Authors": ["authors"],
    "Year": ["year", "publication", "publication year"],
    "DOI": ["doi"],
    "Abstract": ["abstract"],
    "Journal Name": ["source title", "journal name"],
    "Journal Area": [
        "research areas",
        "web of science categories",
        "category",
        "subject area"
    ],
    "ISSN Code": ["issn", "issn code"],
    "eISSN Code": ["eissn", "eissn code"],
    "Document Type": ["document type", "doctype"]
}

# ----------------------------------------------------------
# EN — 7️⃣ BUILD STANDARDIZED DATAFRAME
# PT — 7️⃣ CONSTRUIR DATAFRAME PADRONIZADO
# ----------------------------------------------------------

filtered_data = {}

for standard_col, possible_names in column_map.items():

    matched_col = next(
        (col for col in possible_names if col in df_all.columns),
        None
    )

    if matched_col:
        filtered_data[standard_col] = df_all[matched_col]
    else:
        # EN — If column not found, fill with placeholder
        # PT — Se coluna não encontrada, preencher com placeholder
        filtered_data[standard_col] = ["-"] * len(df_all)

wos_filtered = pd.DataFrame(filtered_data)

# EN — Replace missing values with "-"
# PT — Substituir valores ausentes por "-"
wos_filtered.fillna("-", inplace=True)

# ----------------------------------------------------------
# EN — 8️⃣ SAVE CLEAN DATASET
# PT — 8️⃣ SALVAR DATASET LIMPO
# ----------------------------------------------------------

wos_filtered.to_csv(OUTPUT_PATH, index=False, encoding="utf-8")

# ----------------------------------------------------------
# EN — 9️⃣ SUMMARY REPORT
# PT — 9️⃣ RELATÓRIO RESUMIDO
# ----------------------------------------------------------

print("\n✅ EN — Clean file saved successfully.")
print("✅ PT — Arquivo limpo salvo com sucesso.")

print(f"\n📍 EN — Output path: {OUTPUT_PATH}")
print(f"📍 PT — Caminho de saída: {OUTPUT_PATH}")

print(f"\n📊 EN — Final shape: {wos_filtered.shape[0]} rows × {wos_filtered.shape[1]} columns")
print(f"📊 PT — Dimensão final: {wos_filtered.shape[0]} linhas × {wos_filtered.shape[1]} colunas")

print("\n📑 EN — Final columns:")
print("📑 PT — Colunas finais:")
for col in wos_filtered.columns:
    print(" -", col)


📂 EN — 15 Excel file(s) found.
📂 PT — 15 arquivo(s) Excel encontrado(s).

📄 EN — Loading: savedrecs (1).xls
📄 PT — Carregando: savedrecs (1).xls
📄 EN — Loading: savedrecs (10).xls
📄 PT — Carregando: savedrecs (10).xls
📄 EN — Loading: savedrecs (11).xls
📄 PT — Carregando: savedrecs (11).xls
📄 EN — Loading: savedrecs (12).xls
📄 PT — Carregando: savedrecs (12).xls
📄 EN — Loading: savedrecs (13).xls
📄 PT — Carregando: savedrecs (13).xls
📄 EN — Loading: savedrecs (14).xls
📄 PT — Carregando: savedrecs (14).xls
📄 EN — Loading: savedrecs (2).xls
📄 PT — Carregando: savedrecs (2).xls
📄 EN — Loading: savedrecs (3).xls
📄 PT — Carregando: savedrecs (3).xls
📄 EN — Loading: savedrecs (4).xls
📄 PT — Carregando: savedrecs (4).xls
📄 EN — Loading: savedrecs (5).xls
📄 PT — Carregando: savedrecs (5).xls
📄 EN — Loading: savedrecs (6).xls
📄 PT — Carregando: savedrecs (6).xls
📄 EN — Loading: savedrecs (7).xls
📄 PT — Carregando: savedrecs (7).xls
📄 EN — Loading: savedrecs (8).xls
📄 PT — Carregando: savedrecs (

In [ ]:
# ==========================================================
# EN — STEP 4: EXTRACT AND STANDARDIZE ARXIV DATA
# PT — ETAPA 4: EXTRAIR E PADRONIZAR DADOS DO ARXIV
# ==========================================================


# ----------------------------------------------------------
# EN — 1️⃣ INSTALL REQUIRED DEPENDENCIES (IF NECESSARY)
# PT — 1️⃣ INSTALAR DEPENDÊNCIAS NECESSÁRIAS (SE PRECISO)
# ----------------------------------------------------------

try:
    import feedparser
except ImportError:
    !pip install feedparser -q
    import feedparser

try:
    from bs4 import BeautifulSoup
except ImportError:
    !pip install beautifulsoup4 -q
    from bs4 import BeautifulSoup


# ----------------------------------------------------------
# EN — 2️⃣ IMPORT SCIENTIFIC LIBRARIES
# PT — 2️⃣ IMPORTAR BIBLIOTECAS CIENTÍFICAS
# ----------------------------------------------------------

import requests
import pandas as pd
import time
from pathlib import Path
from datetime import datetime


# ----------------------------------------------------------
# EN — 3️⃣ SEARCH PARAMETERS (REPRODUCIBLE CONFIGURATION)
# PT — 3️⃣ PARÂMETROS DE BUSCA (CONFIGURAÇÃO REPRODUZÍVEL)
# ----------------------------------------------------------

SEARCH_QUERY = 'cat:cs* AND all:"bias" AND all:"LLM" AND all:"behavior"'
BASE_URL = "http://export.arxiv.org/api/query?"

START_DATE = datetime(2018, 1, 1)
END_DATE   = datetime(2026, 1, 12)

MAX_RESULTS = 10000
RESULTS_PER_CALL = 100


# ----------------------------------------------------------
# EN — 4️⃣ DEFINE OUTPUT DIRECTORY
# PT — 4️⃣ DEFINIR DIRETÓRIO DE SAÍDA
# ----------------------------------------------------------

PROJECT_ROOT = Path("/content/drive/MyDrive/Tese/ArticleSLR")
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)

OUTPUT_PATH = PROJECT_ROOT / "arxiv.clean.csv"


# ----------------------------------------------------------
# EN — 5️⃣ FUNCTION: EXTRACT DOI FROM ARXIV PAGE (SCRAPING)
# PT — 5️⃣ FUNÇÃO: EXTRAIR DOI DA PÁGINA DO ARXIV (SCRAPING)
# ----------------------------------------------------------

def extract_doi_from_arxiv(arxiv_id):
    """
    EN — Extract DOI directly from https://arxiv.org/abs/{arxiv_id}
    PT — Extrai o DOI diretamente da página https://arxiv.org/abs/{arxiv_id}
    """
    url = f"https://arxiv.org/abs/{arxiv_id}"
    headers = {"User-Agent": "Mozilla/5.0"}

    try:
        response = requests.get(url, headers=headers, timeout=10)
        if response.status_code != 200:
            return "-"

        soup = BeautifulSoup(response.text, "html.parser")
        doi_tag = soup.find("a", href=lambda x: x and "doi.org" in x)

        return doi_tag.text.strip() if doi_tag else "-"

    except Exception:
        return "-"


# ----------------------------------------------------------
# EN — 6️⃣ COLLECT ARTICLES (PAGINATED API REQUESTS)
# PT — 6️⃣ COLETAR ARTIGOS (REQUISIÇÕES PAGINADAS)
# ----------------------------------------------------------

entries = []

print("🌐 EN — Querying arXiv API...")
print("🌐 PT — Consultando API do arXiv...")

for start in range(0, MAX_RESULTS, RESULTS_PER_CALL):

    query = (
        f"search_query={SEARCH_QUERY}"
        f"&start={start}"
        f"&max_results={RESULTS_PER_CALL}"
        f"&sortBy=submittedDate"
        f"&sortOrder=descending"
    )

    response = requests.get(BASE_URL + query)
    feed = feedparser.parse(response.text)

    if not feed.entries:
        break

    entries.extend(feed.entries)

    # EN — Respect arXiv API rate limits
    # PT — Respeitar limite de requisições da API
    time.sleep(1)


# ----------------------------------------------------------
# EN — 7️⃣ FILTER BY PUBLICATION DATE
# PT — 7️⃣ FILTRAR POR DATA DE PUBLICAÇÃO
# ----------------------------------------------------------

filtered_entries = []

for entry in entries:
    published_date = datetime.strptime(entry.published[:10], "%Y-%m-%d")

    if START_DATE <= published_date <= END_DATE:
        filtered_entries.append(entry)

print(f"📅 EN — Articles after date filtering: {len(filtered_entries)}")
print(f"📅 PT — Artigos após filtro temporal: {len(filtered_entries)}")


# ----------------------------------------------------------
# EN — 8️⃣ PROCESS METADATA AND BUILD DATASET
# PT — 8️⃣ PROCESSAR METADADOS E CONSTRUIR DATASET
# ----------------------------------------------------------

data = []

print("🔎 EN — Extracting DOIs from arXiv pages...")
print("🔎 PT — Extraindo DOIs das páginas do arXiv...")

for i, entry in enumerate(filtered_entries, 1):

    title = entry.title.strip().replace("\n", " ")
    authors = ", ".join(author.name for author in entry.authors)
    year = entry.published.split("-")[0]
    abstract = entry.summary.strip().replace("\n", " ")
    categories = entry.tags[0]["term"] if entry.tags else "-"
    journal = entry.get("arxiv_journal_ref", "-")
    doc_type = "Preprint"

    arxiv_id = entry.id.split("/")[-1]
    doi = extract_doi_from_arxiv(arxiv_id)

    data.append({
        "Title": title,
        "Authors": authors,
        "Year": year,
        "DOI": doi,
        "Abstract": abstract,
        "Journal Name": journal,
        "Journal Area": categories,
        "ISSN Code": "-",
        "eISSN Code": "-",
        "Document Type": doc_type
    })

    # EN — Friendly rate limit to avoid blocking
    # PT — Intervalo amigável para evitar bloqueio
    time.sleep(0.5)


# ----------------------------------------------------------
# EN — 9️⃣ CREATE FINAL DATAFRAME
# PT — 9️⃣ CRIAR DATAFRAME FINAL
# ----------------------------------------------------------

arxiv_df = pd.DataFrame(data)
arxiv_df.fillna("-", inplace=True)


# ----------------------------------------------------------
# EN — 🔟 SAVE CLEAN DATASET
# PT — 🔟 SALVAR DATASET LIMPO
# ----------------------------------------------------------

arxiv_df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8")


# ----------------------------------------------------------
# EN — 1️⃣1️⃣ SUMMARY REPORT
# PT — 1️⃣1️⃣ RELATÓRIO RESUMIDO
# ----------------------------------------------------------

print("\n✅ EN — Extraction completed successfully.")
print("✅ PT — Extração concluída com sucesso.")

print(f"\n📍 EN — Output path: {OUTPUT_PATH}")
print(f"📍 PT — Caminho de saída: {OUTPUT_PATH}")

print(f"\n📊 EN — Final shape: {arxiv_df.shape[0]} rows × {arxiv_df.shape[1]} columns")
print(f"📊 PT — Dimensão final: {arxiv_df.shape[0]} linhas × {arxiv_df.shape[1]} colunas")

print("\n📑 EN — Final columns:")
print("📑 PT — Colunas finais:")
for col in arxiv_df.columns:
    print(" -", col)


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 3.0 MB/s eta 0:00:00
🌐 EN — Querying arXiv API...
🌐 PT — Consultando API do arXiv...
📅 EN — Articles after date filtering: 346
📅 PT — Artigos após filtro temporal: 346
🔎 EN — Extracting DOIs from arXiv pages...
🔎 PT — Extraindo DOIs das páginas do arXiv...

✅ EN — Extraction completed successfully.
✅ PT — Extração concluída com sucesso.

📍 EN — Output path: /content/drive/MyDrive/Tese/ArticleSLR/arxiv.clean.csv
📍 PT — Caminho de saída: /content/drive/MyDrive/Tese/ArticleSLR/arxiv.clean.csv

📊 EN — Final shape: 346 rows × 10 columns
📊 PT — Dimensão final: 346 linhas × 10 colunas

📑 EN — Final columns:
📑 PT — Colunas finais:
 - Title
 - Authors
 - Year
 - DOI
 - Abstract
 - Journal Name
 - Journal Area
 - ISSN Code
 - eISSN Code
 - Document Type


In [ ]:
# ==========================================================
# EN — STEP 5: UNIFY SCOPUS + WEB OF SCIENCE + ARXIV DATA
# PT — ETAPA 5: UNIFICAR DADOS DO SCOPUS + WEB OF SCIENCE + ARXIV
# ==========================================================

import pandas as pd
from pathlib import Path

# ----------------------------------------------------------
# EN — 1️⃣ DEFINE PROJECT PATHS AND INPUT FILES
# PT — 1️⃣ DEFINIR CAMINHOS DO PROJETO E ARQUIVOS DE ENTRADA
# ----------------------------------------------------------

PROJECT_ROOT = Path("/content/drive/MyDrive/Tese/ArticleSLR")

SCOPUS_FILE = PROJECT_ROOT / "scopus.clean.csv"
WOS_FILE    = PROJECT_ROOT / "wos.clean.csv"
ARXIV_FILE  = PROJECT_ROOT / "arxiv.clean.csv"

OUTPUT_FILE = PROJECT_ROOT / "data1.unification.csv"


# ----------------------------------------------------------
# EN — 2️⃣ VALIDATE INPUT FILES EXISTENCE
# PT — 2️⃣ VALIDAR EXISTÊNCIA DOS ARQUIVOS DE ENTRADA
# ----------------------------------------------------------

for file_path in [SCOPUS_FILE, WOS_FILE, ARXIV_FILE]:
    if not file_path.exists():
        raise FileNotFoundError(
            f"❌ EN — File not found: {file_path}\n"
            f"❌ PT — Arquivo não encontrado: {file_path}"
        )


# ----------------------------------------------------------
# EN — 3️⃣ LOAD CLEAN DATASETS
# PT — 3️⃣ CARREGAR DATASETS LIMPOS
# ----------------------------------------------------------

df_scopus = pd.read_csv(SCOPUS_FILE)
df_wos    = pd.read_csv(WOS_FILE)
df_arxiv  = pd.read_csv(ARXIV_FILE)

print("📥 EN — Datasets loaded successfully.")
print("📥 PT — Datasets carregados com sucesso.\n")


# ----------------------------------------------------------
# EN — 4️⃣ ADD SOURCE IDENTIFICATION COLUMN
# PT — 4️⃣ ADICIONAR COLUNA DE IDENTIFICAÇÃO DA FONTE
# ----------------------------------------------------------

df_scopus["Data"] = "Scopus"
df_wos["Data"]    = "Web of Science"
df_arxiv["Data"]  = "arXiv"


# ----------------------------------------------------------
# EN — 5️⃣ ENSURE COMMON COLUMN STRUCTURE
# PT — 5️⃣ GARANTIR ESTRUTURA COMUM DE COLUNAS
# ----------------------------------------------------------

common_columns = list(
    set(df_scopus.columns) &
    set(df_wos.columns) &
    set(df_arxiv.columns)
)

df_scopus = df_scopus[common_columns]
df_wos    = df_wos[common_columns]
df_arxiv  = df_arxiv[common_columns]


# ----------------------------------------------------------
# EN — 6️⃣ CONCATENATE ALL DATASETS
# PT — 6️⃣ CONCATENAR TODOS OS DATASETS
# ----------------------------------------------------------

df_merged = pd.concat(
    [df_scopus, df_wos, df_arxiv],
    ignore_index=True
)

# EN — Replace missing values with "-"
# PT — Substituir valores ausentes por "-"
df_merged.fillna("-", inplace=True)


# ----------------------------------------------------------
# EN — 7️⃣ REORDER COLUMNS (SCIENTIFIC STANDARD)
# PT — 7️⃣ REORDENAR COLUNAS (PADRÃO CIENTÍFICO)
# ----------------------------------------------------------

desired_order = [
    "Title",
    "Authors",
    "Year",
    "DOI",
    "Journal Name",
    "Journal Area",
    "Abstract",
    "ISSN Code",
    "eISSN Code",
    "Document Type",
    "Data"
]

final_columns = [col for col in desired_order if col in df_merged.columns]
df_merged = df_merged[final_columns]


# ----------------------------------------------------------
# EN — 8️⃣ SAVE UNIFIED DATASET
# PT — 8️⃣ SALVAR DATASET UNIFICADO
# ----------------------------------------------------------

df_merged.to_csv(OUTPUT_FILE, index=False, encoding="utf-8")


# ----------------------------------------------------------
# EN — 9️⃣ SUMMARY REPORT
# PT — 9️⃣ RELATÓRIO RESUMIDO
# ----------------------------------------------------------

total = len(df_merged)
total_scopus = len(df_scopus)
total_wos = len(df_wos)
total_arxiv = len(df_arxiv)

print("\n✅ EN — Unified dataset saved successfully.")
print("✅ PT — Dataset unificado salvo com sucesso.")

print(f"\n📍 EN — Output path: {OUTPUT_FILE}")
print(f"📍 PT — Caminho de saída: {OUTPUT_FILE}")

print(f"\n📊 EN — Total combined records: {total}")
print(f"📊 PT — Total de registros combinados: {total}")

print(f"\n   • EN — Scopus records: {total_scopus}")
print(f"   • PT — Registros Scopus: {total_scopus}")

print(f"   • EN — Web of Science records: {total_wos}")
print(f"   • PT — Registros Web of Science: {total_wos}")

print(f"   • EN — arXiv records: {total_arxiv}")
print(f"   • PT — Registros arXiv: {total_arxiv}")

print("\n📑 EN — Final column order:")
print("📑 PT — Ordem final das colunas:")
for col in df_merged.columns:
    print(" -", col)


📥 EN — Datasets loaded successfully.
📥 PT — Datasets carregados com sucesso.


✅ EN — Unified dataset saved successfully.
✅ PT — Dataset unificado salvo com sucesso.

📍 EN — Output path: /content/drive/MyDrive/Tese/ArticleSLR/data1.unification.csv
📍 PT — Caminho de saída: /content/drive/MyDrive/Tese/ArticleSLR/data1.unification.csv

📊 EN — Total combined records: 23331
📊 PT — Total de registros combinados: 23331

   • EN — Scopus records: 8380
   • PT — Registros Scopus: 8380
   • EN — Web of Science records: 14605
   • PT — Registros Web of Science: 14605
   • EN — arXiv records: 346
   • PT — Registros arXiv: 346

📑 EN — Final column order:
📑 PT — Ordem final das colunas:
 - Title
 - Authors
 - Year
 - DOI
 - Journal Name
 - Journal Area
 - Abstract
 - ISSN Code
 - eISSN Code
 - Document Type
 - Data


In [ ]:
# ==========================================================
# EN — STEP 6: DETECT DUPLICATED ARTICLES
# PT — ETAPA 6: DETECTAR ARTIGOS DUPLICADOS
# ==========================================================

import pandas as pd
from pathlib import Path


# ----------------------------------------------------------
# EN — 1️⃣ DEFINE PROJECT PATHS AND INPUT FILE
# PT — 1️⃣ DEFINIR CAMINHOS DO PROJETO E ARQUIVO DE ENTRADA
# ----------------------------------------------------------

PROJECT_ROOT = Path("/content/drive/MyDrive/Tese/ArticleSLR")

INPUT_FILE = PROJECT_ROOT / "data1.unification.csv"
OUTPUT_FILE = PROJECT_ROOT / "data0.duplicated.csv"


# ----------------------------------------------------------
# EN — 2️⃣ VALIDATE INPUT FILE
# PT — 2️⃣ VALIDAR ARQUIVO DE ENTRADA
# ----------------------------------------------------------

if not INPUT_FILE.exists():
    raise FileNotFoundError(
        f"❌ EN — File not found: {INPUT_FILE}\n"
        f"❌ PT — Arquivo não encontrado: {INPUT_FILE}"
    )


# ----------------------------------------------------------
# EN — 3️⃣ LOAD UNIFIED DATASET
# PT — 3️⃣ CARREGAR DATASET UNIFICADO
# ----------------------------------------------------------

df = pd.read_csv(INPUT_FILE)

print("📥 EN — Unified dataset loaded successfully.")
print("📥 PT — Dataset unificado carregado com sucesso.\n")


# ----------------------------------------------------------
# EN — 4️⃣ NORMALIZE COMPARISON COLUMNS
# PT — 4️⃣ NORMALIZAR COLUNAS PARA COMPARAÇÃO
# ----------------------------------------------------------

# EN — Normalize title, DOI and journal name for safe comparison
# PT — Normalizar título, DOI e nome do periódico para comparação segura

df["title_norm"] = df["Title"].astype(str).str.strip().str.lower()
df["doi_norm"] = df["DOI"].fillna("-").astype(str).str.strip().str.lower()
df["journal_norm"] = df["Journal Name"].fillna("-").astype(str).str.strip().str.lower()


# ----------------------------------------------------------
# EN — 5️⃣ DETECT DUPLICATED TITLES
# PT — 5️⃣ DETECTAR TÍTULOS DUPLICADOS
# ----------------------------------------------------------

# EN — Group by normalized title and keep groups with more than one record
# PT — Agrupar por título normalizado e manter grupos com mais de um registro

duplicated_groups = df.groupby("title_norm").filter(lambda x: len(x) > 1)


# ----------------------------------------------------------
# EN — 6️⃣ PREPARE EXPORT (REMOVE AUXILIARY COLUMNS)
# PT — 6️⃣ PREPARAR EXPORTAÇÃO (REMOVER COLUNAS AUXILIARES)
# ----------------------------------------------------------

duplicated_export = duplicated_groups.drop(
    columns=["title_norm", "doi_norm", "journal_norm"]
)


# ----------------------------------------------------------
# EN — 7️⃣ SAVE DUPLICATED RECORDS
# PT — 7️⃣ SALVAR REGISTROS DUPLICADOS
# ----------------------------------------------------------

duplicated_export.to_csv(OUTPUT_FILE, index=False, encoding="utf-8")


# ----------------------------------------------------------
# EN — 8️⃣ SUMMARY REPORT (SCIENTIFIC LOGGING)
# PT — 8️⃣ RELATÓRIO RESUMIDO (LOG CIENTÍFICO)
# ----------------------------------------------------------

num_unique_duplicated_titles = duplicated_groups["title_norm"].nunique()
num_duplicated_rows = len(duplicated_export)
total_records = len(df)

print("\n🔍 EN — Duplicate detection completed.")
print("🔍 PT — Detecção de duplicados concluída.")

print(f"\n📊 EN — Total records analyzed: {total_records}")
print(f"📊 PT — Total de registros analisados: {total_records}")

print(f"\n📄 EN — Number of duplicated titles: {num_unique_duplicated_titles}")
print(f"📄 PT — Número de títulos duplicados: {num_unique_duplicated_titles}")

print(f"\n📑 EN — Total duplicated entries (rows): {num_duplicated_rows}")
print(f"📑 PT — Total de entradas duplicadas (linhas): {num_duplicated_rows}")

print(f"\n📤 EN — Duplicates file saved at: {OUTPUT_FILE}")
print(f"📤 PT — Arquivo de duplicados salvo em: {OUTPUT_FILE}")


📥 EN — Unified dataset loaded successfully.
📥 PT — Dataset unificado carregado com sucesso.


🔍 EN — Duplicate detection completed.
🔍 PT — Detecção de duplicados concluída.

📊 EN — Total records analyzed: 23331
📊 PT — Total de registros analisados: 23331

📄 EN — Number of duplicated titles: 3997
📄 PT — Número de títulos duplicados: 3997

📑 EN — Total duplicated entries (rows): 7998
📑 PT — Total de entradas duplicadas (linhas): 7998

📤 EN — Duplicates file saved at: /content/drive/MyDrive/Tese/ArticleSLR/data0.duplicated.csv
📤 PT — Arquivo de duplicados salvo em: /content/drive/MyDrive/Tese/ArticleSLR/data0.duplicated.csv


In [ ]:
# ==========================================================
# EN — STEP 7: REMOVE DUPLICATED ARTICLES
# PT — ETAPA 7: REMOVER ARTIGOS DUPLICADOS
# ==========================================================

import pandas as pd
from pathlib import Path


# ----------------------------------------------------------
# EN — 1️⃣ DEFINE PROJECT PATHS
# PT — 1️⃣ DEFINIR CAMINHOS DO PROJETO
# ----------------------------------------------------------

PROJECT_ROOT = Path("/content/drive/MyDrive/Tese/ArticleSLR")

INPUT_FILE = PROJECT_ROOT / "data1.unification.csv"
OUTPUT_FILE = PROJECT_ROOT / "data2.unique.csv"


# ----------------------------------------------------------
# EN — 2️⃣ VALIDATE INPUT FILE
# PT — 2️⃣ VALIDAR ARQUIVO DE ENTRADA
# ----------------------------------------------------------

if not INPUT_FILE.exists():
    raise FileNotFoundError(
        f"❌ EN — File not found: {INPUT_FILE}\n"
        f"❌ PT — Arquivo não encontrado: {INPUT_FILE}"
    )


# ----------------------------------------------------------
# EN — 3️⃣ LOAD UNIFIED DATASET
# PT — 3️⃣ CARREGAR DATASET UNIFICADO
# ----------------------------------------------------------

df = pd.read_csv(INPUT_FILE)

print("📥 EN — Unified dataset loaded successfully.")
print("📥 PT — Dataset unificado carregado com sucesso.\n")


# ----------------------------------------------------------
# EN — 4️⃣ NORMALIZE TITLE FOR DUPLICATE DETECTION
# PT — 4️⃣ NORMALIZAR TÍTULO PARA DETECÇÃO DE DUPLICIDADE
# ----------------------------------------------------------

# EN — Create normalized title column (lowercase + strip)
# PT — Criar coluna de título normalizado (minúsculo + remover espaços)

df["title_norm"] = (
    df["Title"]
    .astype(str)
    .str.strip()
    .str.lower()
)


# ----------------------------------------------------------
# EN — 5️⃣ REMOVE DUPLICATES (KEEP FIRST OCCURRENCE)
# PT — 5️⃣ REMOVER DUPLICATAS (MANTER PRIMEIRA OCORRÊNCIA)
# ----------------------------------------------------------

original_total = len(df)

df_unique = df.drop_duplicates(
    subset="title_norm",
    keep="first"
)

unique_total = len(df_unique)
removed_total = original_total - unique_total


# ----------------------------------------------------------
# EN — 6️⃣ REMOVE AUXILIARY COLUMN
# PT — 6️⃣ REMOVER COLUNA AUXILIAR
# ----------------------------------------------------------

df_unique = df_unique.drop(columns=["title_norm"])


# ----------------------------------------------------------
# EN — 7️⃣ SAVE CLEAN UNIQUE DATASET
# PT — 7️⃣ SALVAR DATASET LIMPO E ÚNICO
# ----------------------------------------------------------

df_unique.to_csv(OUTPUT_FILE, index=False, encoding="utf-8")


# ----------------------------------------------------------
# EN — 8️⃣ SCIENTIFIC SUMMARY REPORT
# PT — 8️⃣ RELATÓRIO CIENTÍFICO RESUMIDO
# ----------------------------------------------------------

print("\n✅ EN — Unique dataset saved successfully.")
print("✅ PT — Dataset único salvo com sucesso.")

print(f"\n📊 EN — Original total records: {original_total}")
print(f"📊 PT — Total original de registros: {original_total}")

print(f"\n🧹 EN — Duplicates removed: {removed_total}")
print(f"🧹 PT — Duplicatas removidas: {removed_total}")

print(f"\n📄 EN — Remaining unique records: {unique_total}")
print(f"📄 PT — Registros únicos restantes: {unique_total}")

print(f"\n📍 EN — Output path: {OUTPUT_FILE}")
print(f"📍 PT — Caminho de saída: {OUTPUT_FILE}")


📥 EN — Unified dataset loaded successfully.
📥 PT — Dataset unificado carregado com sucesso.


✅ EN — Unique dataset saved successfully.
✅ PT — Dataset único salvo com sucesso.

📊 EN — Original total records: 23331
📊 PT — Total original de registros: 23331

🧹 EN — Duplicates removed: 4001
🧹 PT — Duplicatas removidas: 4001

📄 EN — Remaining unique records: 19330
📄 PT — Registros únicos restantes: 19330

📍 EN — Output path: /content/drive/MyDrive/Tese/ArticleSLR/data2.unique.csv
📍 PT — Caminho de saída: /content/drive/MyDrive/Tese/ArticleSLR/data2.unique.csv


In [ ]:
# ==========================================================
# EN — EXTRA ANALYSIS 1: IDENTIFY UNIQUE JOURNALS
# PT — ANÁLISE EXTRA 1: IDENTIFICAR REVISTAS ÚNICAS
# ==========================================================

import pandas as pd
from pathlib import Path


# ----------------------------------------------------------
# EN — 1️⃣ DEFINE PROJECT PATHS
# PT — 1️⃣ DEFINIR CAMINHOS DO PROJETO
# ----------------------------------------------------------

PROJECT_ROOT = Path("/content/drive/MyDrive/Tese/ArticleSLR")

INPUT_FILE = PROJECT_ROOT / "data2.unique.csv"
OUTPUT_FILE = PROJECT_ROOT / "data0.journals.csv"


# ----------------------------------------------------------
# EN — 2️⃣ VALIDATE INPUT FILE
# PT — 2️⃣ VALIDAR ARQUIVO DE ENTRADA
# ----------------------------------------------------------

if not INPUT_FILE.exists():
    raise FileNotFoundError(
        f"❌ EN — File not found: {INPUT_FILE}\n"
        f"❌ PT — Arquivo não encontrado: {INPUT_FILE}"
    )


# ----------------------------------------------------------
# EN — 3️⃣ LOAD UNIQUE ARTICLES DATASET
# PT — 3️⃣ CARREGAR DATASET DE ARTIGOS ÚNICOS
# ----------------------------------------------------------

df = pd.read_csv(INPUT_FILE)

print("📥 EN — Dataset loaded successfully.")
print("📥 PT — Dataset carregado com sucesso.\n")


# ----------------------------------------------------------
# EN — 4️⃣ NORMALIZE COLUMN NAMES
# PT — 4️⃣ NORMALIZAR NOMES DAS COLUNAS
# ----------------------------------------------------------

df.columns = df.columns.str.strip().str.lower()


# ----------------------------------------------------------
# EN — 5️⃣ MAP JOURNAL-RELATED COLUMNS
# PT — 5️⃣ MAPEAR COLUNAS RELACIONADAS A PERIÓDICOS
# ----------------------------------------------------------

column_map = {
    "Journal Name": ["journal name", "source title"],
    "Journal Area": ["journal area", "subject area", "category"],
    "ISSN Code": ["issn code", "issn"],
    "eISSN Code": ["eissn code", "eissn"]
}

df_mapped = pd.DataFrame()

for target_col, options in column_map.items():

    matched_col = next((opt for opt in options if opt in df.columns), None)

    if matched_col:
        df_mapped[target_col] = df[matched_col]
    else:
        # EN — If column not found, fill with placeholder
        # PT — Se coluna não encontrada, preencher com placeholder
        df_mapped[target_col] = "-"


# ----------------------------------------------------------
# EN — 6️⃣ NORMALIZE JOURNAL NAME (FOR DEDUPLICATION)
# PT — 6️⃣ NORMALIZAR NOME DA REVISTA (PARA DEDUPLICAÇÃO)
# ----------------------------------------------------------

df_mapped["Journal Name"] = (
    df_mapped["Journal Name"]
    .astype(str)
    .str.upper()
    .str.strip()
)


# ----------------------------------------------------------
# EN — 7️⃣ REMOVE DUPLICATED JOURNALS
# PT — 7️⃣ REMOVER REVISTAS DUPLICADAS
# ----------------------------------------------------------

df_unique_journals = (
    df_mapped
    .drop_duplicates(subset="Journal Name")
    .reset_index(drop=True)
)


# ----------------------------------------------------------
# EN — 8️⃣ SAVE JOURNAL LIST
# PT — 8️⃣ SALVAR LISTA DE REVISTAS
# ----------------------------------------------------------

df_unique_journals.to_csv(OUTPUT_FILE, index=False, encoding="utf-8")


# ----------------------------------------------------------
# EN — 9️⃣ SUMMARY REPORT
# PT — 9️⃣ RELATÓRIO RESUMIDO
# ----------------------------------------------------------

total_journals = df_unique_journals.shape[0]

print("\n✅ EN — Journal list generated successfully.")
print("✅ PT — Lista de revistas gerada com sucesso.")

print(f"\n📚 EN — Total unique journals: {total_journals}")
print(f"📚 PT — Total de revistas únicas: {total_journals}")

print(f"\n📍 EN — Output path: {OUTPUT_FILE}")
print(f"📍 PT — Caminho de saída: {OUTPUT_FILE}")

print("\n📑 EN — Columns in journal dataset:")
print("📑 PT — Colunas no dataset de revistas:")
for col in df_unique_journals.columns:
    print(" -", col)


📥 EN — Dataset loaded successfully.
📥 PT — Dataset carregado com sucesso.


✅ EN — Journal list generated successfully.
✅ PT — Lista de revistas gerada com sucesso.

📚 EN — Total unique journals: 2532
📚 PT — Total de revistas únicas: 2532

📍 EN — Output path: /content/drive/MyDrive/Tese/ArticleSLR/data0.journals.csv
📍 PT — Caminho de saída: /content/drive/MyDrive/Tese/ArticleSLR/data0.journals.csv

📑 EN — Columns in journal dataset:
📑 PT — Colunas no dataset de revistas:
 - Journal Name
 - Journal Area
 - ISSN Code
 - eISSN Code


In [ ]:
# ==========================================================
# EN — STEP 8: EXTRACT AND STANDARDIZE JCR DATA
# PT — ETAPA 8: EXTRAIR E PADRONIZAR DADOS DO JCR
# ==========================================================

import pandas as pd
from pathlib import Path
from glob import glob


# ----------------------------------------------------------
# EN — 1️⃣ DEFINE PROJECT AND JCR DIRECTORIES
# PT — 1️⃣ DEFINIR DIRETÓRIOS DO PROJETO E DO JCR
# ----------------------------------------------------------

PROJECT_ROOT = Path("/content/drive/MyDrive/Tese/ArticleSLR")
JCR_DIR = PROJECT_ROOT / "JCR_Data"
OUTPUT_FILE = PROJECT_ROOT / "jcr1.clean.csv"


# ----------------------------------------------------------
# EN — 2️⃣ VALIDATE JCR DIRECTORY
# PT — 2️⃣ VALIDAR DIRETÓRIO DO JCR
# ----------------------------------------------------------

if not JCR_DIR.exists():
    raise FileNotFoundError(
        f"❌ EN — JCR directory not found: {JCR_DIR}\n"
        f"❌ PT — Diretório JCR não encontrado: {JCR_DIR}"
    )


# ----------------------------------------------------------
# EN — 3️⃣ DEFINE STANDARD COLUMN MAPPING
# PT — 3️⃣ DEFINIR MAPEAMENTO PADRÃO DE COLUNAS
# ----------------------------------------------------------

column_map = {
    "Journal Name": ["source title", "journal", "journal name"],
    "Journal Area": [
        "research areas",
        "web of science categories",
        "category",
        "subject area",
        "areas"
    ],
    "ISSN Code": ["issn", "issn code"],
    "eISSN Code": ["eissn", "eissn code"],
    "Impact": ["jif quartile", "5 year jif quartile", "quartile"],
    "Publisher": ["publisher"],
    "Country": ["country"]
}


# ----------------------------------------------------------
# EN — 4️⃣ SEARCH FOR JCR FILES (CSV + EXCEL)
# PT — 4️⃣ BUSCAR ARQUIVOS JCR (CSV + EXCEL)
# ----------------------------------------------------------

jcr_files = (
    glob(str(JCR_DIR / "*.csv")) +
    glob(str(JCR_DIR / "*.xls*"))
)

if not jcr_files:
    print("⚠️ EN — No JCR files found.")
    print("⚠️ PT — Nenhum arquivo JCR encontrado.")


# ----------------------------------------------------------
# EN — 5️⃣ PROCESS EACH JCR FILE
# PT — 5️⃣ PROCESSAR CADA ARQUIVO JCR
# ----------------------------------------------------------

jcr_dataframes = []

for file_path in jcr_files:

    file_path = Path(file_path)

    print(f"\n📄 EN — Processing file: {file_path.name}")
    print(f"📄 PT — Processando arquivo: {file_path.name}")

    try:
        # EN — Read CSV or Excel file accordingly
        # PT — Ler arquivo CSV ou Excel conforme extensão
        if file_path.suffix.lower() == ".csv":
            df = pd.read_csv(file_path)
        elif file_path.suffix.lower() == ".xlsx":
            df = pd.read_excel(file_path, engine="openpyxl")
        else:
            df = pd.read_excel(file_path)

        # EN — Normalize column names
        # PT — Normalizar nomes das colunas
        df.columns = df.columns.str.strip().str.lower()

        filtered_data = {}

        # EN — Extract standardized columns based on mapping
        # PT — Extrair colunas padronizadas com base no mapeamento
        for standard_col, possible_names in column_map.items():

            matched_col = next(
                (col for col in possible_names if col in df.columns),
                None
            )

            if matched_col:
                filtered_data[standard_col] = df[matched_col]
            else:
                filtered_data[standard_col] = ["-"] * len(df)

        jcr_dataframes.append(pd.DataFrame(filtered_data))

    except Exception as e:
        print(f"⚠️ EN — Error processing file: {file_path.name}")
        print(f"⚠️ PT — Erro ao processar arquivo: {file_path.name}")
        print(f"   ➜ {e}")


# ----------------------------------------------------------
# EN — 6️⃣ CONCATENATE ALL JCR DATA
# PT — 6️⃣ CONCATENAR TODOS OS DADOS JCR
# ----------------------------------------------------------

if jcr_dataframes:

    jcr_combined = pd.concat(jcr_dataframes, ignore_index=True)
    jcr_combined.fillna("-", inplace=True)

    # ------------------------------------------------------
    # EN — 7️⃣ SAVE CLEAN JCR DATASET
    # PT — 7️⃣ SALVAR DATASET JCR LIMPO
    # ------------------------------------------------------

    jcr_combined.to_csv(OUTPUT_FILE, index=False, encoding="utf-8")

    # ------------------------------------------------------
    # EN — 8️⃣ SUMMARY REPORT
    # PT — 8️⃣ RELATÓRIO RESUMIDO
    # ------------------------------------------------------

    print("\n✅ EN — JCR dataset saved successfully.")
    print("✅ PT — Dataset JCR salvo com sucesso.")

    print(f"\n📍 EN — Output path: {OUTPUT_FILE}")
    print(f"📍 PT — Caminho de saída: {OUTPUT_FILE}")

    print(f"\n📊 EN — Final shape: {jcr_combined.shape[0]} rows × {jcr_combined.shape[1]} columns")
    print(f"📊 PT — Dimensão final: {jcr_combined.shape[0]} linhas × {jcr_combined.shape[1]} colunas")

    print("\n📑 EN — Final columns:")
    print("📑 PT — Colunas finais:")
    for col in jcr_combined.columns:
        print(" -", col)

else:
    print("\n⚠️ EN — No valid JCR data extracted.")
    print("⚠️ PT — Nenhum dado válido do JCR foi extraído.")



📄 EN — Processing file: JCR_Enriched_With_ISSN_Matching.csv
📄 PT — Processando arquivo: JCR_Enriched_With_ISSN_Matching.csv

📄 EN — Processing file: JCR_business_business.finance_27JAN.xlsx
📄 PT — Processando arquivo: JCR_business_business.finance_27JAN.xlsx

📄 EN — Processing file: JCR_ComputerScience_27JAN.xlsx
📄 PT — Processando arquivo: JCR_ComputerScience_27JAN.xlsx

✅ EN — JCR dataset saved successfully.
✅ PT — Dataset JCR salvo com sucesso.

📍 EN — Output path: /content/drive/MyDrive/Tese/ArticleSLR/jcr1.clean.csv
📍 PT — Caminho de saída: /content/drive/MyDrive/Tese/ArticleSLR/jcr1.clean.csv

📊 EN — Final shape: 21737 rows × 7 columns
📊 PT — Dimensão final: 21737 linhas × 7 colunas

📑 EN — Final columns:
📑 PT — Colunas finais:
 - Journal Name
 - Journal Area
 - ISSN Code
 - eISSN Code
 - Impact
 - Publisher
 - Country


In [ ]:
# ==========================================================
# EN — STEP 9: REMOVE JCR DUPLICATES WITH PRIORITY RULES
# PT — ETAPA 9: REMOVER DUPLICATAS DO JCR COM REGRAS DE PRIORIDADE
# ==========================================================

import pandas as pd
from pathlib import Path


# ----------------------------------------------------------
# EN — 1️⃣ DEFINE PROJECT PATHS
# PT — 1️⃣ DEFINIR CAMINHOS DO PROJETO
# ----------------------------------------------------------

PROJECT_ROOT = Path("/content/drive/MyDrive/Tese/ArticleSLR")

INPUT_FILE = PROJECT_ROOT / "jcr1.clean.csv"
OUTPUT_FILE = PROJECT_ROOT / "jcr2.unique.csv"


# ----------------------------------------------------------
# EN — 2️⃣ VALIDATE INPUT FILE
# PT — 2️⃣ VALIDAR ARQUIVO DE ENTRADA
# ----------------------------------------------------------

if not INPUT_FILE.exists():
    raise FileNotFoundError(
        f"❌ EN — File not found: {INPUT_FILE}\n"
        f"❌ PT — Arquivo não encontrado: {INPUT_FILE}"
    )


# ----------------------------------------------------------
# EN — 3️⃣ LOAD JCR DATASET
# PT — 3️⃣ CARREGAR DATASET DO JCR
# ----------------------------------------------------------

df = pd.read_csv(INPUT_FILE)

print("📥 EN — JCR dataset loaded successfully.")
print("📥 PT — Dataset JCR carregado com sucesso.\n")


# ----------------------------------------------------------
# EN — 4️⃣ STANDARDIZE CRITICAL COLUMNS
# PT — 4️⃣ PADRONIZAR COLUNAS CRÍTICAS
# ----------------------------------------------------------

df.columns = df.columns.str.strip()

df["Journal Name"] = (
    df["Journal Name"]
    .astype(str)
    .str.upper()
    .str.strip()
)

df["Journal Area"] = (
    df["Journal Area"]
    .fillna("-")
    .astype(str)
    .str.strip()
)

df["Impact"] = (
    df["Impact"]
    .fillna("-")
    .astype(str)
    .str.upper()
    .str.strip()
)


# ----------------------------------------------------------
# EN — 5️⃣ DEFINE PRIORITY FUNCTION
# PT — 5️⃣ DEFINIR FUNÇÃO DE PRIORIDADE
# ----------------------------------------------------------

def prioridade(row):
    """
    EN — Priority hierarchy:
         0 → Exact match: Business
         1 → Contains 'business'
         2 → Q1
         3 → Q2
         4 → Q3
         5 → Q4
         6 → Others
    PT — Hierarquia de prioridade:
         0 → Exatamente 'Business'
         1 → Contém 'business'
         2 → Q1
         3 → Q2
         4 → Q3
         5 → Q4
         6 → Outros casos
    """

    area = row["Journal Area"].lower()
    impact = row["Impact"].upper()

    if area == "business":
        return 0
    elif "business" in area:
        return 1
    elif impact == "Q1":
        return 2
    elif impact == "Q2":
        return 3
    elif impact == "Q3":
        return 4
    elif impact == "Q4":
        return 5
    else:
        return 6


# ----------------------------------------------------------
# EN — 6️⃣ APPLY PRIORITY RULE
# PT — 6️⃣ APLICAR REGRA DE PRIORIDADE
# ----------------------------------------------------------

df["__priority__"] = df.apply(prioridade, axis=1)


# ----------------------------------------------------------
# EN — 7️⃣ SORT BY JOURNAL AND PRIORITY
# PT — 7️⃣ ORDENAR POR PERIÓDICO E PRIORIDADE
# ----------------------------------------------------------

df_sorted = df.sort_values(
    by=["Journal Name", "__priority__"]
)


# ----------------------------------------------------------
# EN — 8️⃣ KEEP BEST ENTRY PER JOURNAL
# PT — 8️⃣ MANTER MELHOR REGISTRO POR PERIÓDICO
# ----------------------------------------------------------

df_unique = (
    df_sorted
    .drop_duplicates(subset="Journal Name", keep="first")
    .copy()
)

df_unique.drop(columns=["__priority__"], inplace=True)

df_unique.fillna("-", inplace=True)


# ----------------------------------------------------------
# EN — 9️⃣ SAVE CLEAN JCR DATASET
# PT — 9️⃣ SALVAR DATASET JCR LIMPO
# ----------------------------------------------------------

df_unique.to_csv(OUTPUT_FILE, index=False, encoding="utf-8")


# ----------------------------------------------------------
# EN — 🔟 SCIENTIFIC SUMMARY REPORT
# PT — 🔟 RELATÓRIO CIENTÍFICO RESUMIDO
# ----------------------------------------------------------

print("\n✅ EN — JCR deduplication completed successfully.")
print("✅ PT — Deduplicação do JCR concluída com sucesso.")

print(f"\n📉 EN — Total unique journals: {df_unique.shape[0]}")
print(f"📉 PT — Total de periódicos únicos: {df_unique.shape[0]}")

print(f"\n📍 EN — Output path: {OUTPUT_FILE}")
print(f"📍 PT — Caminho de saída: {OUTPUT_FILE}")

print("\n📑 EN — Preserved columns:")
print("📑 PT — Colunas preservadas:")
for col in df_unique.columns:
    print(" -", col)


📥 EN — JCR dataset loaded successfully.
📥 PT — Dataset JCR carregado com sucesso.


✅ EN — JCR deduplication completed successfully.
✅ PT — Deduplicação do JCR concluída com sucesso.

📉 EN — Total unique journals: 20390
📉 PT — Total de periódicos únicos: 20390

📍 EN — Output path: /content/drive/MyDrive/Tese/ArticleSLR/jcr2.unique.csv
📍 PT — Caminho de saída: /content/drive/MyDrive/Tese/ArticleSLR/jcr2.unique.csv

📑 EN — Preserved columns:
📑 PT — Colunas preservadas:
 - Journal Name
 - Journal Area
 - ISSN Code
 - eISSN Code
 - Impact
 - Publisher
 - Country


In [ ]:
# ==========================================================
# EN — STEP 10: MERGE ARTICLES WITH JCR IMPACT DATA
# PT — ETAPA 10: INTEGRAR ARTIGOS COM DADOS DE IMPACTO DO JCR
# ==========================================================

import pandas as pd
from pathlib import Path
import unicodedata


# ----------------------------------------------------------
# EN — 1️⃣ DEFINE PROJECT PATHS
# PT — 1️⃣ DEFINIR CAMINHOS DO PROJETO
# ----------------------------------------------------------

PROJECT_ROOT = Path("/content/drive/MyDrive/Tese/ArticleSLR")

ARTICLES_FILE = PROJECT_ROOT / "data2.unique.csv"
JCR_FILE = PROJECT_ROOT / "jcr2.unique.csv"
OUTPUT_FILE = PROJECT_ROOT / "data3.impact.csv"


# ----------------------------------------------------------
# EN — 2️⃣ VALIDATE INPUT FILES
# PT — 2️⃣ VALIDAR ARQUIVOS DE ENTRADA
# ----------------------------------------------------------

for file_path in [ARTICLES_FILE, JCR_FILE]:
    if not file_path.exists():
        raise FileNotFoundError(
            f"❌ EN — File not found: {file_path}\n"
            f"❌ PT — Arquivo não encontrado: {file_path}"
        )


# ----------------------------------------------------------
# EN — 3️⃣ TEXT NORMALIZATION FUNCTION (ROBUST MATCHING)
# PT — 3️⃣ FUNÇÃO DE NORMALIZAÇÃO DE TEXTO (MATCH ROBUSTO)
# ----------------------------------------------------------

def normalize_text(s):
    """
    EN — Normalize text removing accents, forcing uppercase and trimming spaces
    PT — Normalizar texto removendo acentos, forçando maiúsculas e removendo espaços
    """
    return (
        unicodedata.normalize("NFKD", str(s))
        .encode("ascii", errors="ignore")
        .decode("utf-8")
        .upper()
        .strip()
    )


# ----------------------------------------------------------
# EN — 4️⃣ LOAD DATASETS
# PT — 4️⃣ CARREGAR DATASETS
# ----------------------------------------------------------

articles_raw = pd.read_csv(ARTICLES_FILE)
jcr_raw = pd.read_csv(JCR_FILE)

print("📥 EN — Input datasets loaded successfully.")
print("📥 PT — Datasets de entrada carregados com sucesso.\n")


# ----------------------------------------------------------
# EN — 5️⃣ STANDARDIZE ARTICLES DATA
# PT — 5️⃣ PADRONIZAR DADOS DOS ARTIGOS
# ----------------------------------------------------------

articles_raw.columns = articles_raw.columns.str.strip().str.lower()

column_map_articles = {
    "Title": ["title"],
    "Authors": ["authors"],
    "Year": ["year", "publication"],
    "DOI": ["doi"],
    "Abstract": ["abstract"],
    "Journal Name": ["journal name", "source title"],
    "Document Type": ["document type", "doctype"],
    "Data": ["data"]
}

def extract_articles(df, column_map):

    out = {}

    for std, variants in column_map.items():

        matched_col = next(
            (v for v in variants if v in df.columns),
            None
        )

        if matched_col:
            out[std] = df[matched_col].fillna("-").astype(str)
        else:
            out[std] = ["-"] * len(df)

    return pd.DataFrame(out)


articles_df = extract_articles(articles_raw, column_map_articles)

# EN — Normalize journal name for matching
# PT — Normalizar nome do periódico para realizar o merge
articles_df["Journal Name"] = articles_df["Journal Name"].apply(normalize_text)


# ----------------------------------------------------------
# EN — 6️⃣ STANDARDIZE JCR DATA (AUTHORITATIVE SOURCE)
# PT — 6️⃣ PADRONIZAR DADOS DO JCR (FONTE OFICIAL)
# ----------------------------------------------------------

jcr_raw.columns = jcr_raw.columns.str.strip().str.lower()

jcr_df = jcr_raw.rename(columns={
    "journal name": "Journal Name",
    "journal area": "Journal Area",
    "subject area": "Journal Area",
    "category": "Journal Area",
    "issn": "ISSN Code",
    "issn code": "ISSN Code",
    "eissn": "eISSN Code",
    "eissn code": "eISSN Code",
    "impact": "Impact",
    "5 year jif quartile": "Impact",
    "jif quartile": "Impact",
    "publisher": "Publisher",
    "country": "Country"
})

# EN — Normalize journal name in JCR as well
# PT — Normalizar nome do periódico também no JCR
jcr_df["Journal Name"] = jcr_df["Journal Name"].apply(normalize_text)

# EN — Remove duplicate journal entries in JCR
# PT — Remover duplicatas de periódicos no JCR
jcr_df = jcr_df.drop_duplicates(subset="Journal Name", keep="first").copy()


# ----------------------------------------------------------
# EN — 7️⃣ PERFORM LEFT MERGE (JCR COMPLEMENTS ARTICLES)
# PT — 7️⃣ REALIZAR MERGE À ESQUERDA (JCR COMPLEMENTA ARTIGOS)
# ----------------------------------------------------------

merged_df = pd.merge(
    articles_df,
    jcr_df[
        [
            "Journal Name",
            "Journal Area",
            "ISSN Code",
            "eISSN Code",
            "Impact",
            "Publisher",
            "Country"
        ]
    ],
    how="left",
    on="Journal Name"
)

merged_df.fillna("-", inplace=True)


# ----------------------------------------------------------
# EN — 8️⃣ FINAL COLUMN ORGANIZATION
# PT — 8️⃣ ORGANIZAÇÃO FINAL DAS COLUNAS
# ----------------------------------------------------------

final_columns = [
    "Title",
    "Authors",
    "Year",
    "DOI",
    "Abstract",
    "Journal Name",
    "Journal Area",
    "ISSN Code",
    "eISSN Code",
    "Document Type",
    "Impact",
    "Publisher",
    "Country",
    "Data"
]

final_df = merged_df[final_columns].copy()


# ----------------------------------------------------------
# EN — 9️⃣ SAVE FINAL IMPACT-ENRICHED DATASET
# PT — 9️⃣ SALVAR DATASET FINAL COM IMPACTO
# ----------------------------------------------------------

final_df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8")


# ----------------------------------------------------------
# EN — 🔟 SCIENTIFIC SUMMARY REPORT
# PT — 🔟 RELATÓRIO CIENTÍFICO RESUMIDO
# ----------------------------------------------------------

print("\n✅ EN — Impact merge completed successfully.")
print("✅ PT — Integração com impacto concluída com sucesso.")

print(f"\n📊 EN — Total final articles: {final_df.shape[0]}")
print(f"📊 PT — Total final de artigos: {final_df.shape[0]}")

print(f"\n📍 EN — Output path: {OUTPUT_FILE}")
print(f"📍 PT — Caminho de saída: {OUTPUT_FILE}")

print("\n📑 EN — Final columns:")
print("📑 PT — Colunas finais:")
for col in final_df.columns:
    print(" -", col)


📥 EN — Input datasets loaded successfully.
📥 PT — Datasets de entrada carregados com sucesso.


✅ EN — Impact merge completed successfully.
✅ PT — Integração com impacto concluída com sucesso.

📊 EN — Total final articles: 19330
📊 PT — Total final de artigos: 19330

📍 EN — Output path: /content/drive/MyDrive/Tese/ArticleSLR/data3.impact.csv
📍 PT — Caminho de saída: /content/drive/MyDrive/Tese/ArticleSLR/data3.impact.csv

📑 EN — Final columns:
📑 PT — Colunas finais:
 - Title
 - Authors
 - Year
 - DOI
 - Abstract
 - Journal Name
 - Journal Area
 - ISSN Code
 - eISSN Code
 - Document Type
 - Impact
 - Publisher
 - Country
 - Data


In [ ]:
# ==========================================================
# EN — EXTRA ANALYSIS 2: NUMBER OF ARTICLES PER IMPACT FACTOR
# PT — ANÁLISE EXTRA 2: QUANTIDADE DE ARTIGOS POR FATOR DE IMPACTO
# ==========================================================

import pandas as pd
from pathlib import Path


# ----------------------------------------------------------
# EN — 1️⃣ DEFINE PROJECT PATH
# PT — 1️⃣ DEFINIR CAMINHO DO PROJETO
# ----------------------------------------------------------

PROJECT_ROOT = Path("/content/drive/MyDrive/Tese/ArticleSLR")
INPUT_FILE = PROJECT_ROOT / "data3.impact.csv"


# ----------------------------------------------------------
# EN — 2️⃣ VALIDATE INPUT FILE
# PT — 2️⃣ VALIDAR ARQUIVO DE ENTRADA
# ----------------------------------------------------------

if not INPUT_FILE.exists():
    raise FileNotFoundError(
        f"❌ EN — File not found: {INPUT_FILE}\n"
        f"❌ PT — Arquivo não encontrado: {INPUT_FILE}"
    )


# ----------------------------------------------------------
# EN — 3️⃣ LOAD DATASET
# PT — 3️⃣ CARREGAR DATASET
# ----------------------------------------------------------

df = pd.read_csv(INPUT_FILE)

print("📥 EN — Impact dataset loaded successfully.")
print("📥 PT — Dataset com impacto carregado com sucesso.\n")


# ----------------------------------------------------------
# EN — 4️⃣ STANDARDIZE COLUMN NAMES
# PT — 4️⃣ PADRONIZAR NOMES DAS COLUNAS
# ----------------------------------------------------------

df.columns = df.columns.str.strip().str.lower()


# ----------------------------------------------------------
# EN — 5️⃣ VALIDATE 'IMPACT' COLUMN
# PT — 5️⃣ VALIDAR EXISTÊNCIA DA COLUNA 'IMPACT'
# ----------------------------------------------------------

if "impact" not in df.columns:

    print("❌ EN — Column 'Impact' not found in dataset.")
    print("❌ PT — Coluna 'Impact' não foi encontrada no dataset.")

else:

    # ------------------------------------------------------
    # EN — 6️⃣ STANDARDIZE IMPACT VALUES
    # PT — 6️⃣ PADRONIZAR VALORES DE IMPACTO
    # ------------------------------------------------------

    df["impact"] = (
        df["impact"]
        .fillna("-")
        .astype(str)
        .str.upper()
        .str.strip()
    )

    # ------------------------------------------------------
    # EN — 7️⃣ COUNT ARTICLES PER IMPACT CATEGORY
    # PT — 7️⃣ CONTAR ARTIGOS POR CATEGORIA DE IMPACTO
    # ------------------------------------------------------

    impact_counts = (
        df["impact"]
        .value_counts()
        .sort_index()
    )

    total_articles = df.shape[0]


    # ------------------------------------------------------
    # EN — 8️⃣ SCIENTIFIC REPORT
    # PT — 8️⃣ RELATÓRIO CIENTÍFICO
    # ------------------------------------------------------

    print("📊 EN — Number of articles per Impact category:\n")
    print("📊 PT — Quantidade de artigos por categoria de Impacto:\n")

    for impact, count in impact_counts.items():
        percentage = (count / total_articles) * 100
        print(f" - {impact:<5}: {count} articles ({percentage:.2f}%)")

    print(f"\n🧮 EN — Total articles analyzed: {total_articles}")
    print(f"🧮 PT — Total de artigos analisados: {total_articles}")


📥 EN — Impact dataset loaded successfully.
📥 PT — Dataset com impacto carregado com sucesso.

📊 EN — Number of articles per Impact category:

📊 PT — Quantidade de artigos por categoria de Impacto:

 - -    : 4858 articles (25.13%)
 - Q1   : 7332 articles (37.93%)
 - Q2   : 4714 articles (24.39%)
 - Q3   : 1625 articles (8.41%)
 - Q4   : 801 articles (4.14%)

🧮 EN — Total articles analyzed: 19330
🧮 PT — Total de artigos analisados: 19330


In [ ]:
# ==========================================================
# EN — STEP 11: BUILD FINAL DATASET WITH ONLY Q1 AND Q2 JOURNALS
# PT — ETAPA 11: CONSTRUIR BASE FINAL APENAS COM PERIÓDICOS Q1 E Q2
# ==========================================================

import pandas as pd
from pathlib import Path


# ----------------------------------------------------------
# EN — 1️⃣ DEFINE PROJECT PATHS
# PT — 1️⃣ DEFINIR CAMINHOS DO PROJETO
# ----------------------------------------------------------

PROJECT_ROOT = Path("/content/drive/MyDrive/Tese/ArticleSLR")

INPUT_FILE = PROJECT_ROOT / "data3.impact.csv"
OUTPUT_FILE = PROJECT_ROOT / "data4.q1q2.csv"


# ----------------------------------------------------------
# EN — 2️⃣ VALIDATE INPUT FILE
# PT — 2️⃣ VALIDAR ARQUIVO DE ENTRADA
# ----------------------------------------------------------

if not INPUT_FILE.exists():
    raise FileNotFoundError(
        f"❌ EN — File not found: {INPUT_FILE}\n"
        f"❌ PT — Arquivo não encontrado: {INPUT_FILE}"
    )


# ----------------------------------------------------------
# EN — 3️⃣ LOAD FULL IMPACT DATASET
# PT — 3️⃣ CARREGAR DATASET COMPLETO COM IMPACTO
# ----------------------------------------------------------

df = pd.read_csv(INPUT_FILE)

print("📥 EN — Impact dataset loaded successfully.")
print("📥 PT — Dataset com impacto carregado com sucesso.\n")


# ----------------------------------------------------------
# EN — 4️⃣ VALIDATE 'Impact' COLUMN
# PT — 4️⃣ VALIDAR EXISTÊNCIA DA COLUNA 'Impact'
# ----------------------------------------------------------

if "Impact" not in df.columns:
    raise ValueError(
        "❌ EN — Column 'Impact' does not exist in dataset.\n"
        "❌ PT — A coluna 'Impact' não existe no dataset."
    )


# ----------------------------------------------------------
# EN — 5️⃣ STANDARDIZE IMPACT COLUMN
# PT — 5️⃣ PADRONIZAR COLUNA IMPACT
# ----------------------------------------------------------

df["Impact"] = (
    df["Impact"]
    .astype(str)
    .str.strip()
    .str.upper()
)


# ----------------------------------------------------------
# EN — 6️⃣ FILTER ONLY Q1 AND Q2 JOURNALS
# PT — 6️⃣ FILTRAR APENAS PERIÓDICOS Q1 E Q2
# ----------------------------------------------------------

filtered_df = df[
    df["Impact"].isin(["Q1", "Q2"])
].copy()

filtered_df.fillna("-", inplace=True)


# ----------------------------------------------------------
# EN — 7️⃣ SAVE FINAL FILTERED DATASET
# PT — 7️⃣ SALVAR DATASET FINAL FILTRADO
# ----------------------------------------------------------

filtered_df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8")


# ----------------------------------------------------------
# EN — 8️⃣ SCIENTIFIC SUMMARY REPORT
# PT — 8️⃣ RELATÓRIO CIENTÍFICO RESUMIDO
# ----------------------------------------------------------

total_original = df.shape[0]
total_filtered = filtered_df.shape[0]
percentage_retained = (total_filtered / total_original) * 100 if total_original > 0 else 0

print("\n✅ EN — Q1/Q2 dataset created successfully.")
print("✅ PT — Dataset Q1/Q2 criado com sucesso.")

print(f"\n📊 EN — Original total articles: {total_original}")
print(f"📊 PT — Total original de artigos: {total_original}")

print(f"\n🏆 EN — Articles in Q1/Q2 journals: {total_filtered}")
print(f"🏆 PT — Artigos em periódicos Q1/Q2: {total_filtered}")

print(f"\n📈 EN — Percentage retained: {percentage_retained:.2f}%")
print(f"📈 PT — Percentual mantido: {percentage_retained:.2f}%")

print(f"\n📍 EN — Output path: {OUTPUT_FILE}")
print(f"📍 PT — Caminho de saída: {OUTPUT_FILE}")

print("\n📑 EN — Columns preserved:")
print("📑 PT — Colunas mantidas:")
for col in filtered_df.columns:
    print(" -", col)


📥 EN — Impact dataset loaded successfully.
📥 PT — Dataset com impacto carregado com sucesso.


✅ EN — Q1/Q2 dataset created successfully.
✅ PT — Dataset Q1/Q2 criado com sucesso.

📊 EN — Original total articles: 19330
📊 PT — Total original de artigos: 19330

🏆 EN — Articles in Q1/Q2 journals: 12046
🏆 PT — Artigos em periódicos Q1/Q2: 12046

📈 EN — Percentage retained: 62.32%
📈 PT — Percentual mantido: 62.32%

📍 EN — Output path: /content/drive/MyDrive/Tese/ArticleSLR/data4.q1q2.csv
📍 PT — Caminho de saída: /content/drive/MyDrive/Tese/ArticleSLR/data4.q1q2.csv

📑 EN — Columns preserved:
📑 PT — Colunas mantidas:
 - Title
 - Authors
 - Year
 - DOI
 - Abstract
 - Journal Name
 - Journal Area
 - ISSN Code
 - eISSN Code
 - Document Type
 - Impact
 - Publisher
 - Country
 - Data


In [ ]:
# ==========================================================
# EN — STEP 12: ABSTRACT SCREENING BASED ON THREE-PILLAR RULE
# PT — ETAPA 12: TRIAGEM DE ABSTRACT COM REGRA DOS TRÊS PILARES
# ==========================================================

import pandas as pd
from pathlib import Path


# ----------------------------------------------------------
# EN — 1️⃣ DEFINE PROJECT PATHS
# PT — 1️⃣ DEFINIR CAMINHOS DO PROJETO
# ----------------------------------------------------------

PROJECT_ROOT = Path("/content/drive/MyDrive/Tese/ArticleSLR")

INPUT_FILE = PROJECT_ROOT / "data4.q1q2.csv"
OUTPUT_INCLUDED = PROJECT_ROOT / "data5.pillar_included.csv"
OUTPUT_EXCLUDED = PROJECT_ROOT / "data0.pillar_excluded.csv"


# ----------------------------------------------------------
# EN — 2️⃣ VALIDATE INPUT FILE
# PT — 2️⃣ VALIDAR ARQUIVO DE ENTRADA
# ----------------------------------------------------------

if not INPUT_FILE.exists():
    raise FileNotFoundError(
        f"❌ EN — File not found: {INPUT_FILE}\n"
        f"❌ PT — Arquivo não encontrado: {INPUT_FILE}"
    )


# ----------------------------------------------------------
# EN — 3️⃣ LOAD DATASET
# PT — 3️⃣ CARREGAR DATASET
# ----------------------------------------------------------

df = pd.read_csv(INPUT_FILE)

print("📥 EN — Dataset loaded successfully.")
print("📥 PT — Dataset carregado com sucesso.\n")


# ----------------------------------------------------------
# EN — 4️⃣ VALIDATE ABSTRACT COLUMN
# PT — 4️⃣ VALIDAR COLUNA ABSTRACT
# ----------------------------------------------------------

if "Abstract" not in df.columns:
    raise ValueError(
        "❌ EN — Column 'Abstract' not found in dataset.\n"
        "❌ PT — Coluna 'Abstract' não encontrada no dataset."
    )


# ----------------------------------------------------------
# EN — 5️⃣ NORMALIZE ABSTRACT TEXT
# PT — 5️⃣ NORMALIZAR TEXTO DO ABSTRACT
# ----------------------------------------------------------

df["Abstract_norm"] = (
    df["Abstract"]
    .fillna("")
    .astype(str)
    .str.lower()
)


# ----------------------------------------------------------
# EN — 6️⃣ PILLARS BASED ON SEARCH STRING KEYWORDS
# PT — 6️⃣ PILARES CONFORME PALAVRAS-CHAVE DOS STRINGS DA BUSCA
# ----------------------------------------------------------

pillar_1 = [
    "algorithm", "artificial intelligence", "ai", "machine learning",
    "computational", "automated",
    "platform algorithm", "recommendation algorithm",
    "search algorithm", "ranking algorithm",
    "algorithmic governance"
]

pillar_2 = [
    "bias", "algorithmic bias", "discriminat", "fairness",
    "algorithmic discrimination", "algorithmic fairness",
    "algorithmic manipulation", "algorithmic persuasion",
    "digital nudging", "ai bias", "language model bias",
    "large language model", "llm", "generative ai",
    "gpt", "bert", "transformer model", "foundation model"
]

pillar_3 = [
    "consumer behavior", "consumer behaviour",
    "user behavior", "user behaviour",
    "online behavior", "search behavior", "search behaviour",
    "decision making", "decision-making",
    "information seeking", "content consumption",
    "platform power", "tech giant", "big tech",
    "google", "facebook", "meta", "amazon", "microsoft",
    "social media platform", "search engine"
]


# ----------------------------------------------------------
# EN — 7️⃣ HELPER FUNCTION: LITERAL SUBSTRING DETECTION
# PT — 7️⃣ FUNÇÃO AUXILIAR: DETECÇÃO LITERAL POR SUBSTRING
# ----------------------------------------------------------

def detect_keywords(text, keyword_list):
    found = []
    for kw in keyword_list:
        if kw in text:
            found.append(kw)
    return found


# ----------------------------------------------------------
# EN — 8️⃣ APPLY SCREENING PER PILLAR
# PT — 8️⃣ APLICAR TRIAGEM POR PILAR
# ----------------------------------------------------------

df["Pillar_1_hits"] = df["Abstract_norm"].apply(
    lambda x: detect_keywords(x, pillar_1)
)

df["Pillar_2_hits"] = df["Abstract_norm"].apply(
    lambda x: detect_keywords(x, pillar_2)
)

df["Pillar_3_hits"] = df["Abstract_norm"].apply(
    lambda x: detect_keywords(x, pillar_3)
)


# ----------------------------------------------------------
# EN — 9️⃣ INCLUSION RULE (AT LEAST ONE HIT PER PILLAR)
# PT — 9️⃣ REGRA DE INCLUSÃO (AO MENOS UM TERMO POR PILAR)
# ----------------------------------------------------------

df["Pillar_1_present"] = df["Pillar_1_hits"].apply(lambda x: len(x) > 0)
df["Pillar_2_present"] = df["Pillar_2_hits"].apply(lambda x: len(x) > 0)
df["Pillar_3_present"] = df["Pillar_3_hits"].apply(lambda x: len(x) > 0)

df["Verdict"] = df.apply(
    lambda row: "INCLUDE"
    if row["Pillar_1_present"]
    and row["Pillar_2_present"]
    and row["Pillar_3_present"]
    else "EXCLUDE",
    axis=1
)


# ----------------------------------------------------------
# EN — 🔟 SPLIT INCLUDED AND EXCLUDED DATASETS
# PT — 🔟 SEPARAR BASES INCLUÍDOS E EXCLUÍDOS
# ----------------------------------------------------------

df_included = df[df["Verdict"] == "INCLUDE"].copy()
df_excluded = df[df["Verdict"] == "EXCLUDE"].copy()


# ----------------------------------------------------------
# EN — 1️⃣1️⃣ REMOVE AUXILIARY COLUMN
# PT — 1️⃣1️⃣ REMOVER COLUNA AUXILIAR
# ----------------------------------------------------------

df_included.drop(columns=["Abstract_norm"], inplace=True)
df_excluded.drop(columns=["Abstract_norm"], inplace=True)


# ----------------------------------------------------------
# EN — 1️⃣2️⃣ SAVE OUTPUT FILES
# PT — 1️⃣2️⃣ SALVAR ARQUIVOS DE SAÍDA
# ----------------------------------------------------------

df_included.to_csv(OUTPUT_INCLUDED, index=False)
df_excluded.to_csv(OUTPUT_EXCLUDED, index=False)


# ----------------------------------------------------------
# EN — 1️⃣3️⃣ SCIENTIFIC SUMMARY REPORT
# PT — 1️⃣3️⃣ RELATÓRIO CIENTÍFICO RESUMIDO
# ----------------------------------------------------------

total_records = len(df)
included_count = len(df_included)
excluded_count = len(df_excluded)

print("\n✅ EN — Screening completed successfully.")
print("✅ PT — Triagem concluída com sucesso.")

print(f"\n📊 EN — Total records analyzed: {total_records}")
print(f"📊 PT — Total de registros analisados: {total_records}")

print(f"\n✔ EN — Included articles: {included_count}")
print(f"✔ PT — Artigos incluídos: {included_count}")

print(f"\n✖ EN — Excluded articles: {excluded_count}")
print(f"✖ PT — Artigos excluídos: {excluded_count}")

print(f"\n📍 EN — Included file: {OUTPUT_INCLUDED}")
print(f"📍 PT — Arquivo de incluídos: {OUTPUT_INCLUDED}")

print(f"\n📍 EN — Excluded file: {OUTPUT_EXCLUDED}")
print(f"📍 PT — Arquivo de excluídos: {OUTPUT_EXCLUDED}")


📥 EN — Dataset loaded successfully.
📥 PT — Dataset carregado com sucesso.


✅ EN — Screening completed successfully.
✅ PT — Triagem concluída com sucesso.

📊 EN — Total records analyzed: 12046
📊 PT — Total de registros analisados: 12046

✔ EN — Included articles: 2944
✔ PT — Artigos incluídos: 2944

✖ EN — Excluded articles: 9102
✖ PT — Artigos excluídos: 9102

📍 EN — Included file: /content/drive/MyDrive/Tese/ArticleSLR/data5.pillar_included.csv
📍 PT — Arquivo de incluídos: /content/drive/MyDrive/Tese/ArticleSLR/data5.pillar_included.csv

📍 EN — Excluded file: /content/drive/MyDrive/Tese/ArticleSLR/data0.pillar_excluded.csv
📍 PT — Arquivo de excluídos: /content/drive/MyDrive/Tese/ArticleSLR/data0.pillar_excluded.csv


In [ ]:
# ==========================================================
# EN — EXTRA ANALYSIS 3: NUMBER OF ARTICLES PER DATABASE SOURCE
# PT — ANÁLISE EXTRA 3: QUANTIDADE DE ARTIGOS POR BASE DE DADOS
# ==========================================================

import pandas as pd
from pathlib import Path


# ----------------------------------------------------------
# EN — 1️⃣ DEFINE PROJECT PATH
# PT — 1️⃣ DEFINIR CAMINHO DO PROJETO
# ----------------------------------------------------------

PROJECT_ROOT = Path("/content/drive/MyDrive/Tese/ArticleSLR")
INPUT_FILE = PROJECT_ROOT / "data5.pillar_included.csv"


# ----------------------------------------------------------
# EN — 2️⃣ VALIDATE INPUT FILE
# PT — 2️⃣ VALIDAR ARQUIVO DE ENTRADA
# ----------------------------------------------------------

if not INPUT_FILE.exists():
    raise FileNotFoundError(
        f"❌ EN — File not found: {INPUT_FILE}\n"
        f"❌ PT — Arquivo não encontrado: {INPUT_FILE}"
    )


# ----------------------------------------------------------
# EN — 3️⃣ LOAD INCLUDED ARTICLES DATASET
# PT — 3️⃣ CARREGAR DATASET DE ARTIGOS INCLUÍDOS
# ----------------------------------------------------------

df = pd.read_csv(INPUT_FILE)

print("📥 EN — Included dataset loaded successfully.")
print("📥 PT — Dataset de incluídos carregado com sucesso.\n")


# ----------------------------------------------------------
# EN — 4️⃣ VALIDATE 'Data' COLUMN (CASE-SENSITIVE)
# PT — 4️⃣ VALIDAR COLUNA 'Data' (SENSÍVEL A MAIÚSCULAS)
# ----------------------------------------------------------

if "Data" not in df.columns:
    raise ValueError(
        f"❌ EN — Column 'Data' not found.\n"
        f"❌ PT — Coluna 'Data' não encontrada.\n"
        f"Available columns / Colunas disponíveis: {list(df.columns)}"
    )


# ----------------------------------------------------------
# EN — 5️⃣ COUNT RECORDS PER DATA SOURCE
# PT — 5️⃣ CONTAR REGISTROS POR BASE DE DADOS
# ----------------------------------------------------------

count_by_data = (
    df["Data"]
    .fillna("MISSING")
    .value_counts()
    .sort_index()
)

total_records = count_by_data.sum()


# ----------------------------------------------------------
# EN — 6️⃣ SCIENTIFIC SUMMARY REPORT
# PT — 6️⃣ RELATÓRIO CIENTÍFICO RESUMIDO
# ----------------------------------------------------------

print("📊 EN — Number of included articles per database source:\n")
print("📊 PT — Número de artigos incluídos por base de dados:\n")

for source, count in count_by_data.items():
    percentage = (count / total_records) * 100 if total_records > 0 else 0
    print(f" - {source:<20} : {count} articles ({percentage:.2f}%)")

print(f"\n🧮 EN — Total included articles: {total_records}")
print(f"🧮 PT — Total de artigos incluídos: {total_records}")


📥 EN — Included dataset loaded successfully.
📥 PT — Dataset de incluídos carregado com sucesso.

📊 EN — Number of included articles per database source:

📊 PT — Número de artigos incluídos por base de dados:

 - Scopus               : 2575 articles (87.47%)
 - Web of Science       : 369 articles (12.53%)

🧮 EN — Total included articles: 2944
🧮 PT — Total de artigos incluídos: 2944


In [ ]:
# ==========================================================
# EN — STEP 13: PDF RECOVERY (API + CACHE + DOI + TITTLE)
# PT — ETAPA 13: RECUPERAÇÃO DE PDF
# ==========================================================


# ----------------------------------------------------------
# EN — 1️⃣ IMPORT REQUIRED LIBRARIES
# PT — 1️⃣ IMPORTAR BIBLIOTECAS NECESSÁRIAS
# ----------------------------------------------------------

import pandas as pd
import requests
import re
import json
import logging
from pathlib import Path
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from tqdm.notebook import tqdm


# ----------------------------------------------------------
# EN — 2️⃣ SILENCE WARNINGS
# PT — 2️⃣ SILENCIAR WARNINGS
# ----------------------------------------------------------

logging.getLogger("urllib3").setLevel(logging.ERROR)


# ----------------------------------------------------------
# EN — 3️⃣ DEFINE PROJECT PATHS
# PT — 3️⃣ DEFINIR CAMINHOS DO PROJETO
# ----------------------------------------------------------

PROJECT_ROOT = Path("/content/drive/MyDrive/Tese/ArticleSLR")

INPUT_FILE = PROJECT_ROOT / "data5.pillar_included.csv"
OUTPUT_FILE = PROJECT_ROOT / "data6.pdf.csv"
CHECKPOINT_FILE = PROJECT_ROOT / "data6.checkpoint.csv"
CACHE_FILE = PROJECT_ROOT / "api_cache.json"

PDF_DIR = PROJECT_ROOT / "PDFs"
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
PDF_DIR.mkdir(parents=True, exist_ok=True)


# ----------------------------------------------------------
# EN — 4️⃣ LOAD DATASET (AUTO-RESUME)
# PT — 4️⃣ CARREGAR DATASET (RETOMADA AUTOMÁTICA)
# ----------------------------------------------------------

if CHECKPOINT_FILE.exists():
    df = pd.read_csv(CHECKPOINT_FILE)
elif OUTPUT_FILE.exists():
    df = pd.read_csv(OUTPUT_FILE)
else:
    df = pd.read_csv(INPUT_FILE)
    df["PDF"] = "No"
    df["Recovery_Method"] = None


# ----------------------------------------------------------
# EN — 5️⃣ LOAD OR INITIALIZE CACHE
# PT — 5️⃣ CARREGAR OU INICIALIZAR CACHE
# ----------------------------------------------------------

if CACHE_FILE.exists():
    with open(CACHE_FILE, "r") as f:
        api_cache = json.load(f)
else:
    api_cache = {}

def save_cache():
    with open(CACHE_FILE, "w") as f:
        json.dump(api_cache, f)


# ----------------------------------------------------------
# EN — 6️⃣ ADAPTIVE METHOD STATISTICS
# PT — 6️⃣ ESTATÍSTICAS ADAPTATIVAS
# ----------------------------------------------------------

method_stats = {
    "Unpaywall": {"success": 0, "attempts": 0},
    "ScienceDirect": {"success": 0, "attempts": 0},
    "Springer": {"success": 0, "attempts": 0},
    "Wiley": {"success": 0, "attempts": 0},
    "CrossRef_DOI": {"success": 0, "attempts": 0},
    "CrossRef_Title": {"success": 0, "attempts": 0},
    "Semantic_DOI": {"success": 0, "attempts": 0},
    "Semantic_Title": {"success": 0, "attempts": 0},
    "arXiv": {"success": 0, "attempts": 0},
}

def get_method_order():
    return sorted(
        method_stats.keys(),
        key=lambda m: (
            method_stats[m]["success"] /
            method_stats[m]["attempts"]
            if method_stats[m]["attempts"] > 0 else 0
        ),
        reverse=True
    )


# ----------------------------------------------------------
# EN — 7️⃣ HELPER FUNCTIONS
# PT — 7️⃣ FUNÇÕES AUXILIARES
# ----------------------------------------------------------

def clean_filename(text):
    text = re.sub(r"[\\/*?\"<>|:]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text[:200]

def clean_doi(doi):
    return str(doi).replace("https://doi.org/", "").replace("http://doi.org/", "").strip()

def extract_arxiv_id(text):
    match = re.search(r'arxiv\.org/(abs|pdf)/([^\s]+)', text)
    if match:
        return match.group(2).replace(".pdf", "")
    return None

def is_valid_pdf(response):
    return response.status_code == 200 and response.content[:4] == b"%PDF"


# ----------------------------------------------------------
# EN — 8️⃣ CONFIGURE SESSION
# PT — 8️⃣ CONFIGURAR SESSÃO
# ----------------------------------------------------------

session = requests.Session()

session.headers.update({
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/120 Safari/537.36",
    "Accept": "application/pdf,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9"
})

retries = Retry(total=3, backoff_factor=1,
                status_forcelist=[500, 502, 503, 504])
session.mount("https://", HTTPAdapter(max_retries=retries))

UNPAYWALL_API = "https://api.unpaywall.org/v2/"
CROSSREF_API = "https://api.crossref.org/works"
SEMANTIC_API = "https://api.semanticscholar.org/graph/v1/paper"
EMAIL = "pamellabandeira@discente.ufg.br"


# ----------------------------------------------------------
# EN — 9️⃣ INTELLIGENT RECOVERY LOOP
# PT — 9️⃣ LOOP INTELIGENTE DE RECUPERAÇÃO
# ----------------------------------------------------------

downloaded = 0
not_found = 0
total_articles = len(df)

with tqdm(total=total_articles, desc="Full Industrial Recovery", unit="article") as pbar:

    for idx, row in df.iterrows():

        title_raw = str(row["Title"])
        doi_raw = str(row["DOI"])
        title = clean_filename(title_raw)
        file_path = PDF_DIR / f"{title}.pdf"

        if row.get("PDF") == "Yes" and file_path.exists():
            pbar.update(1)
            continue

        pdf_url = None
        method_used = None

        for method in get_method_order():

            method_stats[method]["attempts"] += 1

            try:

                # UNPAYWALL
                if method == "Unpaywall" and doi_raw.lower() != "nan":
                    doi = clean_doi(doi_raw)
                    r = session.get(f"{UNPAYWALL_API}{doi}",
                                    params={"email": EMAIL},
                                    timeout=15)
                    if r.status_code == 200:
                        data = r.json()
                        if data.get("best_oa_location"):
                            pdf_url = data["best_oa_location"].get("url_for_pdf")

                # SCIENCEDIRECT
                elif method == "ScienceDirect" and doi_raw.lower() != "nan":
                    doi = clean_doi(doi_raw)
                    r = session.get(f"https://doi.org/{doi}",
                                    allow_redirects=True)
                    match = re.search(r'/pii/([A-Z0-9]+)', r.url)
                    if match:
                        pii = match.group(1)
                        for url in [
                            f"https://www.sciencedirect.com/science/article/pii/{pii}/pdf",
                            f"https://reader.elsevier.com/reader/sd/pii/{pii}"
                        ]:
                            r_test = session.get(url)
                            if is_valid_pdf(r_test):
                                pdf_url = url
                                break

                # SPRINGER
                elif method == "Springer" and doi_raw.lower() != "nan":
                    doi = clean_doi(doi_raw)
                    pdf_url = f"https://link.springer.com/content/pdf/{doi}.pdf"

                # WILEY
                elif method == "Wiley" and doi_raw.lower() != "nan":
                    doi = clean_doi(doi_raw)
                    pdf_url = f"https://onlinelibrary.wiley.com/doi/pdfdirect/{doi}"

                # CROSSREF DOI
                elif method == "CrossRef_DOI" and doi_raw.lower() != "nan":
                    doi = clean_doi(doi_raw)
                    r = session.get(f"{CROSSREF_API}/{doi}")
                    if r.status_code == 200:
                        links = r.json().get("message", {}).get("link", [])
                        for link in links:
                            if link.get("content-type") == "application/pdf":
                                pdf_url = link.get("URL")
                                break

                # CROSSREF TITLE
                elif method == "CrossRef_Title":
                    r = session.get(CROSSREF_API,
                                    params={"query.title": title_raw, "rows": 1})
                    if r.status_code == 200:
                        items = r.json().get("message", {}).get("items", [])
                        if items:
                            found_doi = items[0].get("DOI")
                            if found_doi:
                                pdf_url = f"https://doi.org/{found_doi}"

                # SEMANTIC DOI
                elif method == "Semantic_DOI" and doi_raw.lower() != "nan":
                    doi = clean_doi(doi_raw)
                    r = session.get(f"{SEMANTIC_API}/DOI:{doi}",
                                    params={"fields": "openAccessPdf"})
                    if r.status_code == 200:
                        data = r.json()
                        if data.get("openAccessPdf"):
                            pdf_url = data["openAccessPdf"].get("url")

                # SEMANTIC TITLE
                elif method == "Semantic_Title":
                    r = session.get(f"{SEMANTIC_API}/search",
                                    params={
                                        "query": title_raw,
                                        "limit": 1,
                                        "fields": "openAccessPdf"
                                    })
                    if r.status_code == 200:
                        papers = r.json().get("data", [])
                        if papers and papers[0].get("openAccessPdf"):
                            pdf_url = papers[0]["openAccessPdf"].get("url")

                # ARXIV
                elif method == "arXiv" and "arxiv" in doi_raw.lower():
                    arxiv_id = extract_arxiv_id(doi_raw)
                    if arxiv_id:
                        pdf_url = f"https://arxiv.org/pdf/{arxiv_id}.pdf"

                if pdf_url:
                    r_test = session.get(pdf_url)
                    if is_valid_pdf(r_test):
                        method_stats[method]["success"] += 1
                        method_used = method
                        break

            except:
                continue

        if pdf_url:
            with open(file_path, "wb") as f:
                f.write(r_test.content)
            df.at[idx, "PDF"] = "Yes"
            df.at[idx, "Recovery_Method"] = method_used
            downloaded += 1
        else:
            not_found += 1

        if (idx + 1) % 100 == 0 or (idx + 1) == total_articles:
            df.to_csv(CHECKPOINT_FILE, index=False)
            df.to_csv(OUTPUT_FILE, index=False)
            save_cache()
            print(f"[{idx+1}/{total_articles}] Downloaded: {downloaded} | Not found: {not_found}")

        pbar.update(1)


print("\n===== FINAL SUMMARY =====")
print(f"Total: {total_articles}")
print(f"Downloaded: {downloaded}")
print(f"Not found: {not_found}")


Full Industrial Recovery:   0%|          | 0/2944 [00:00<?, ?article/s]

/tmp/ipython-input-3638756527.py:288: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Springer' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.at[idx, "Recovery_Method"] = method_used


[100/2944] Downloaded: 19 | Not found: 81
[200/2944] Downloaded: 36 | Not found: 164
[300/2944] Downloaded: 56 | Not found: 244
[400/2944] Downloaded: 141 | Not found: 259
[500/2944] Downloaded: 224 | Not found: 276
[600/2944] Downloaded: 295 | Not found: 305
[700/2944] Downloaded: 366 | Not found: 334
[800/2944] Downloaded: 427 | Not found: 373
[900/2944] Downloaded: 510 | Not found: 390
[1000/2944] Downloaded: 582 | Not found: 418
[1100/2944] Downloaded: 657 | Not found: 443
[1200/2944] Downloaded: 757 | Not found: 443
[1300/2944] Downloaded: 857 | Not found: 443
[1400/2944] Downloaded: 957 | Not found: 443
[1500/2944] Downloaded: 1057 | Not found: 443
[1600/2944] Downloaded: 1157 | Not found: 443
[1700/2944] Downloaded: 1257 | Not found: 443
[1800/2944] Downloaded: 1357 | Not found: 443
[1900/2944] Downloaded: 1457 | Not found: 443
[2000/2944] Downloaded: 1557 | Not found: 443
[2100/2944] Downloaded: 1657 | Not found: 443
[2200/2944] Downloaded: 1757 | Not found: 443
[2300/2944] Dow

In [ ]:
# ==========================================================
# EN — STEP 14: GENERATE A DATA WITH ARTICLES WITHOUT PDF
# PT — ETAPA 14: GERAR UMA BASE SOMENTE COM ARTIGOS SEM PDF
# ==========================================================


# ----------------------------------------------------------
# EN — 1️⃣ IMPORT REQUIRED LIBRARIES
# PT — 1️⃣ IMPORTAR BIBLIOTECAS NECESSÁRIAS
# ----------------------------------------------------------

import pandas as pd
from pathlib import Path


# ----------------------------------------------------------
# EN — 2️⃣ DEFINE PROJECT PATHS
# PT — 2️⃣ DEFINIR CAMINHOS DO PROJETO
# ----------------------------------------------------------

PROJECT_ROOT = Path("/content/drive/MyDrive/Tese/ArticleSLR")

INPUT_FILE = PROJECT_ROOT / "data6.pdf.csv"
OUTPUT_FILE = PROJECT_ROOT / "data0.nopdf.csv"


# ----------------------------------------------------------
# EN — 3️⃣ VALIDATE INPUT FILE EXISTENCE
# PT — 3️⃣ VALIDAR EXISTÊNCIA DO ARQUIVO DE ENTRADA
# ----------------------------------------------------------

if not INPUT_FILE.exists():
    raise FileNotFoundError(f"Input file not found: {INPUT_FILE}")

print(f"📂 Input file confirmed: {INPUT_FILE}")


# ----------------------------------------------------------
# EN — 4️⃣ LOAD DATASET
# PT — 4️⃣ CARREGAR DATASET
# ----------------------------------------------------------

df = pd.read_csv(INPUT_FILE)

print(f"📊 Total articles loaded: {len(df)}")


# ----------------------------------------------------------
# EN — 5️⃣ VALIDATE REQUIRED COLUMN
# PT — 5️⃣ VALIDAR COLUNA OBRIGATÓRIA
# ----------------------------------------------------------

if "PDF" not in df.columns:
    raise ValueError("Required column 'PDF' not found in dataset.")


# ----------------------------------------------------------
# EN — 6️⃣ FILTER ARTICLES WITHOUT PDF
# PT — 6️⃣ FILTRAR ARTIGOS QUE NÃO POSSUEM PDF
# ----------------------------------------------------------

not_found_df = df[df["PDF"] == "No"]

total_missing = len(not_found_df)

print(f"📉 Articles without PDF: {total_missing}")


# ----------------------------------------------------------
# EN — 7️⃣ SAVE FILTERED DATASET
# PT — 7️⃣ SALVAR DATASET FILTRADO
# ----------------------------------------------------------

OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)

not_found_df.to_csv(OUTPUT_FILE, index=False)

print(f"💾 Output file saved at: {OUTPUT_FILE}")


# ----------------------------------------------------------
# EN — 8️⃣ FINAL SUMMARY
# PT — 8️⃣ RELATÓRIO FINAL
# ----------------------------------------------------------

recovery_rate = ((len(df) - total_missing) / len(df)) * 100 if len(df) > 0 else 0

print("\n===== SUMMARY =====")
print(f"Total articles: {len(df)}")
print(f"Articles with PDF: {len(df) - total_missing}")
print(f"Articles without PDF: {total_missing}")
print(f"Current recovery rate: {recovery_rate:.2f}%")



📂 Input file confirmed: /content/drive/MyDrive/Tese/ArticleSLR/data6.pdf.csv
📊 Total articles loaded: 2944
📉 Articles without PDF: 443
💾 Output file saved at: /content/drive/MyDrive/Tese/ArticleSLR/data0.nopdf.csv

===== SUMMARY =====
Total articles: 2944
Articles with PDF: 2501
Articles without PDF: 443
Current recovery rate: 84.95%


In [ ]:

#==========================================================
# EN — STEP 15:
# PT — ETAPA 15: RECUPERAÇÃO AVANÇADA (OPENALEX + CORE)
#==========================================================


# ----------------------------------------------------------
# EN — 1️⃣ IMPORT REQUIRED LIBRARIES
# PT — 1️⃣ IMPORTAR BIBLIOTECAS NECESSÁRIAS
# ----------------------------------------------------------

import pandas as pd
import requests
import re
import time
import logging
from pathlib import Path
from tqdm.notebook import tqdm


# ----------------------------------------------------------
# EN — 2️⃣ SILENCE WARNINGS
# PT — 2️⃣ SILENCIAR WARNINGS
# ----------------------------------------------------------

logging.getLogger("urllib3").setLevel(logging.ERROR)


# ----------------------------------------------------------
# EN — 3️⃣ DEFINE PROJECT PATHS
# PT — 3️⃣ DEFINIR CAMINHOS DO PROJETO
# ----------------------------------------------------------

PROJECT_ROOT = Path("/content/drive/MyDrive/Tese/ArticleSLR")

INPUT_FILE = PROJECT_ROOT / "data0.nopdf.csv"
OUTPUT_FILE = PROJECT_ROOT / "data7.pdf.csv"

PDF_DIR = PROJECT_ROOT / "PDFs"

if not INPUT_FILE.exists():
    raise FileNotFoundError(f"Input file not found: {INPUT_FILE}")

df = pd.read_csv(INPUT_FILE)


# ----------------------------------------------------------
# EN — 4️⃣ CONFIGURE SESSION
# PT — 4️⃣ CONFIGURAR SESSÃO
# ----------------------------------------------------------

session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0",
    "Accept": "application/pdf,*/*"
})

OPENALEX_API = "https://api.openalex.org/works"
CROSSREF_API = "https://api.crossref.org/works"
CORE_API = "https://api.core.ac.uk/v3/search/works"


# ----------------------------------------------------------
# EN — 5️⃣ HELPER FUNCTIONS
# PT — 5️⃣ FUNÇÕES AUXILIARES
# ----------------------------------------------------------

def clean_filename(text):
    text = re.sub(r"[\\/*?\"<>|:]", "", str(text))
    text = re.sub(r"\s+", " ", text).strip()
    return text[:200]

def clean_doi(doi):
    return str(doi).replace("https://doi.org/", "").strip()

def is_valid_pdf(response):
    return response.status_code == 200 and response.content[:4] == b"%PDF"


# ----------------------------------------------------------
# EN — 6️⃣ ADVANCED RECOVERY LOOP
# PT — 6️⃣ LOOP DE RECUPERAÇÃO AVANÇADA
# ----------------------------------------------------------

downloaded = 0
not_found = 0

with tqdm(total=len(df), desc="Zotero-Level Recovery", unit="article") as pbar:

    for idx, row in df.iterrows():

        if row.get("PDF") == "Yes":
            pbar.update(1)
            continue

        title_raw = str(row.get("Title", ""))
        doi_raw = str(row.get("DOI", ""))

        title = clean_filename(title_raw)
        file_path = PDF_DIR / f"{title}.pdf"

        if file_path.exists():
            df.at[idx, "PDF"] = "Yes"
            downloaded += 1
            pbar.update(1)
            continue

        pdf_content = None
        method_used = None

        try:

            # ==================================================
            # 1️⃣ OPENALEX BY DOI
            # ==================================================

            if doi_raw.lower() != "nan" and doi_raw.strip() != "":

                doi = clean_doi(doi_raw)

                r = session.get(f"{OPENALEX_API}/https://doi.org/{doi}")

                if r.status_code == 200:
                    data = r.json()
                    if data.get("open_access") and data["open_access"].get("oa_url"):
                        url = data["open_access"]["oa_url"]
                        r2 = session.get(url)
                        if is_valid_pdf(r2):
                            pdf_content = r2.content
                            method_used = "OpenAlex_DOI"

            # ==================================================
            # 2️⃣ OPENALEX BY TITLE
            # ==================================================

            if not pdf_content:

                r = session.get(
                    OPENALEX_API,
                    params={"search": title_raw, "per_page": 1}
                )

                if r.status_code == 200:
                    results = r.json().get("results", [])
                    if results:
                        oa = results[0].get("open_access")
                        if oa and oa.get("oa_url"):
                            url = oa["oa_url"]
                            r2 = session.get(url)
                            if is_valid_pdf(r2):
                                pdf_content = r2.content
                                method_used = "OpenAlex_Title"

            # ==================================================
            # 3️⃣ CROSSREF TITLE SEARCH (EXPANDED)
            # ==================================================

            if not pdf_content:

                r = session.get(
                    CROSSREF_API,
                    params={"query.title": title_raw, "rows": 1}
                )

                if r.status_code == 200:
                    items = r.json().get("message", {}).get("items", [])
                    if items:
                        links = items[0].get("link", [])
                        for link in links:
                            if link.get("content-type") == "application/pdf":
                                r2 = session.get(link.get("URL"))
                                if is_valid_pdf(r2):
                                    pdf_content = r2.content
                                    method_used = "CrossRef_Title"
                                    break

        except:
            pass


        # ==================================================
        # SAVE RESULT
        # ==================================================

        if pdf_content:
            with open(file_path, "wb") as f:
                f.write(pdf_content)

            df.at[idx, "PDF"] = "Yes"
            df.at[idx, "Recovery_Method"] = method_used
            downloaded += 1
        else:
            not_found += 1

        if (idx + 1) % 50 == 0:
            print(f"[{idx+1}/{len(df)}] Downloaded: {downloaded} | Not found: {not_found}")

        pbar.update(1)


# ----------------------------------------------------------
# EN — 7️⃣ SAVE OUTPUT DATASET
# PT — 7️⃣ SALVAR DATASET FINAL
# ----------------------------------------------------------

df.to_csv(OUTPUT_FILE, index=False)

print("\n===== STEP 18 SUMMARY =====")
print(f"Recovered in Step 18: {downloaded}")
print(f"Still not found: {not_found}")
print(f"Output saved at: {OUTPUT_FILE}")



# ----------------------------------------------------------
# EN — 1️⃣ IMPORT REQUIRED LIBRARIES
# PT — 1️⃣ IMPORTAR BIBLIOTECAS NECESSÁRIAS
# ----------------------------------------------------------

import pandas as pd
import requests
import re
import time
import logging
from pathlib import Path
from tqdm.notebook import tqdm


# ----------------------------------------------------------
# EN — 2️⃣ SILENCE WARNINGS
# PT — 2️⃣ SILENCIAR WARNINGS
# ----------------------------------------------------------

logging.getLogger("urllib3").setLevel(logging.ERROR)


# ----------------------------------------------------------
# EN — 3️⃣ DEFINE PROJECT PATHS
# PT — 3️⃣ DEFINIR CAMINHOS DO PROJETO
# ----------------------------------------------------------

PROJECT_ROOT = Path("/content/drive/MyDrive/Tese/ArticleSLR")

INPUT_FILE = PROJECT_ROOT / "data0.nopdf.csv"
OUTPUT_FILE = PROJECT_ROOT / "data7.pdf.csv"

PDF_DIR = PROJECT_ROOT / "PDFs"

if not INPUT_FILE.exists():
    raise FileNotFoundError(f"Input file not found: {INPUT_FILE}")

df = pd.read_csv(INPUT_FILE)


# ----------------------------------------------------------
# EN — 4️⃣ CONFIGURE SESSION
# PT — 4️⃣ CONFIGURAR SESSÃO
# ----------------------------------------------------------

session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0",
    "Accept": "application/pdf,*/*"
})

OPENALEX_API = "https://api.openalex.org/works"
CROSSREF_API = "https://api.crossref.org/works"
CORE_API = "https://api.core.ac.uk/v3/search/works"


# ----------------------------------------------------------
# EN — 5️⃣ HELPER FUNCTIONS
# PT — 5️⃣ FUNÇÕES AUXILIARES
# ----------------------------------------------------------

def clean_filename(text):
    text = re.sub(r"[\\/*?\"<>|:]", "", str(text))
    text = re.sub(r"\s+", " ", text).strip()
    return text[:200]

def clean_doi(doi):
    return str(doi).replace("https://doi.org/", "").strip()

def is_valid_pdf(response):
    return response.status_code == 200 and response.content[:4] == b"%PDF"


# ----------------------------------------------------------
# EN — 6️⃣ ADVANCED RECOVERY LOOP
# PT — 6️⃣ LOOP DE RECUPERAÇÃO AVANÇADA
# ----------------------------------------------------------

downloaded = 0
not_found = 0

with tqdm(total=len(df), desc="Zotero-Level Recovery", unit="article") as pbar:

    for idx, row in df.iterrows():

        if row.get("PDF") == "Yes":
            pbar.update(1)
            continue

        title_raw = str(row.get("Title", ""))
        doi_raw = str(row.get("DOI", ""))

        title = clean_filename(title_raw)
        file_path = PDF_DIR / f"{title}.pdf"

        if file_path.exists():
            df.at[idx, "PDF"] = "Yes"
            downloaded += 1
            pbar.update(1)
            continue

        pdf_content = None
        method_used = None

        try:

            # ==================================================
            # 1️⃣ OPENALEX BY DOI
            # ==================================================

            if doi_raw.lower() != "nan" and doi_raw.strip() != "":

                doi = clean_doi(doi_raw)

                r = session.get(f"{OPENALEX_API}/https://doi.org/{doi}")

                if r.status_code == 200:
                    data = r.json()
                    if data.get("open_access") and data["open_access"].get("oa_url"):
                        url = data["open_access"]["oa_url"]
                        r2 = session.get(url)
                        if is_valid_pdf(r2):
                            pdf_content = r2.content
                            method_used = "OpenAlex_DOI"

            # ==================================================
            # 2️⃣ OPENALEX BY TITLE
            # ==================================================

            if not pdf_content:

                r = session.get(
                    OPENALEX_API,
                    params={"search": title_raw, "per_page": 1}
                )

                if r.status_code == 200:
                    results = r.json().get("results", [])
                    if results:
                        oa = results[0].get("open_access")
                        if oa and oa.get("oa_url"):
                            url = oa["oa_url"]
                            r2 = session.get(url)
                            if is_valid_pdf(r2):
                                pdf_content = r2.content
                                method_used = "OpenAlex_Title"

            # ==================================================
            # 3️⃣ CROSSREF TITLE SEARCH (EXPANDED)
            # ==================================================

            if not pdf_content:

                r = session.get(
                    CROSSREF_API,
                    params={"query.title": title_raw, "rows": 1}
                )

                if r.status_code == 200:
                    items = r.json().get("message", {}).get("items", [])
                    if items:
                        links = items[0].get("link", [])
                        for link in links:
                            if link.get("content-type") == "application/pdf":
                                r2 = session.get(link.get("URL"))
                                if is_valid_pdf(r2):
                                    pdf_content = r2.content
                                    method_used = "CrossRef_Title"
                                    break

        except:
            pass


        # ==================================================
        # SAVE RESULT
        # ==================================================

        if pdf_content:
            with open(file_path, "wb") as f:
                f.write(pdf_content)

            df.at[idx, "PDF"] = "Yes"
            df.at[idx, "Recovery_Method"] = method_used
            downloaded += 1
        else:
            not_found += 1

        if (idx + 1) % 50 == 0:
            print(f"[{idx+1}/{len(df)}] Downloaded: {downloaded} | Not found: {not_found}")

        pbar.update(1)


# ----------------------------------------------------------
# EN — 7️⃣ SAVE OUTPUT DATASET
# PT — 7️⃣ SALVAR DATASET FINAL
# ----------------------------------------------------------

df.to_csv(OUTPUT_FILE, index=False)

print("\n===== STEP 18 SUMMARY =====")
print(f"Recovered in Step 18: {downloaded}")
print(f"Still not found: {not_found}")
print(f"Output saved at: {OUTPUT_FILE}")


Zotero-Level Recovery:   0%|          | 0/443 [00:00<?, ?article/s]

[50/443] Downloaded: 1 | Not found: 49
[150/443] Downloaded: 5 | Not found: 145
[200/443] Downloaded: 8 | Not found: 192
[250/443] Downloaded: 15 | Not found: 235
[300/443] Downloaded: 18 | Not found: 282
[350/443] Downloaded: 19 | Not found: 331
[400/443] Downloaded: 21 | Not found: 379

===== STEP 18 SUMMARY =====
Recovered in Step 18: 22
Still not found: 421
Output saved at: /content/drive/MyDrive/Tese/ArticleSLR/data7.pdf.csv


Zotero-Level Recovery:   0%|          | 0/443 [00:00<?, ?article/s]

[50/443] Downloaded: 1 | Not found: 49
[150/443] Downloaded: 5 | Not found: 145
[200/443] Downloaded: 8 | Not found: 192
[250/443] Downloaded: 15 | Not found: 235
[300/443] Downloaded: 18 | Not found: 282
[350/443] Downloaded: 19 | Not found: 331
[400/443] Downloaded: 21 | Not found: 379

===== STEP 18 SUMMARY =====
Recovered in Step 18: 22
Still not found: 421
Output saved at: /content/drive/MyDrive/Tese/ArticleSLR/data7.pdf.csv


In [ ]:
# ==========================================================
# EN — STEP 16: UPDATE BASE USING data7.pdf.csv
# PT — ETAPA 16: ATUALIZAR BASE COM DADOS DA data7.pdf.csv
# ==========================================================


# ----------------------------------------------------------
# EN — 1️⃣ IMPORT REQUIRED LIBRARIES
# PT — 1️⃣ IMPORTAR BIBLIOTECAS NECESSÁRIAS
# ----------------------------------------------------------

import pandas as pd
from pathlib import Path


# ----------------------------------------------------------
# EN — 2️⃣ DEFINE PROJECT PATHS
# PT — 2️⃣ DEFINIR CAMINHOS DO PROJETO
# ----------------------------------------------------------

PROJECT_ROOT = Path("/content/drive/MyDrive/Tese/ArticleSLR")

BASE_FILE = PROJECT_ROOT / "data6.pdf.csv"
UPDATE_FILE = PROJECT_ROOT / "data7.pdf.csv"
OUTPUT_FILE = PROJECT_ROOT / "data8.pdf.csv"


# ----------------------------------------------------------
# EN — 3️⃣ VALIDATE FILE EXISTENCE
# PT — 3️⃣ VALIDAR EXISTÊNCIA DOS ARQUIVOS
# ----------------------------------------------------------

if not BASE_FILE.exists():
    raise FileNotFoundError(f"Base file not found: {BASE_FILE}")

if not UPDATE_FILE.exists():
    raise FileNotFoundError(f"Update file not found: {UPDATE_FILE}")

print("📂 Files validated successfully.")


# ----------------------------------------------------------
# EN — 4️⃣ LOAD DATASETS
# PT — 4️⃣ CARREGAR DATASETS
# ----------------------------------------------------------

df_base = pd.read_csv(BASE_FILE)
df_update = pd.read_csv(UPDATE_FILE)

print(f"📊 Base dataset size: {len(df_base)}")
print(f"📊 Update dataset size: {len(df_update)}")


# ----------------------------------------------------------
# EN — 5️⃣ VALIDATE REQUIRED COLUMNS
# PT — 5️⃣ VALIDAR COLUNAS NECESSÁRIAS
# ----------------------------------------------------------

required_cols = ["DOI", "PDF"]

for col in required_cols:
    if col not in df_base.columns:
        raise ValueError(f"Column '{col}' not found in base dataset.")
    if col not in df_update.columns:
        raise ValueError(f"Column '{col}' not found in update dataset.")


# ----------------------------------------------------------
# EN — 6️⃣ UPDATE BASE WHERE data7 HAS PDF = Yes
# PT — 6️⃣ ATUALIZAR BASE ONDE data7 TEM PDF = Yes
# ----------------------------------------------------------

updated_count = 0

# EN — Create set of DOIs marked as Yes in update dataset
# PT — Criar conjunto de DOIs com PDF = Yes na base de atualização

update_yes_dois = set(
    df_update.loc[df_update["PDF"] == "Yes", "DOI"]
    .dropna()
    .astype(str)
)

# EN — Update base dataset
# PT — Atualizar base principal

for idx, row in df_base.iterrows():

    doi = str(row["DOI"])

    if doi in update_yes_dois:
        if row["PDF"] != "Yes":
            df_base.at[idx, "PDF"] = "Yes"
            updated_count += 1


# ----------------------------------------------------------
# EN — 7️⃣ SAVE UPDATED DATASET
# PT — 7️⃣ SALVAR DATASET ATUALIZADO
# ----------------------------------------------------------

df_base.to_csv(OUTPUT_FILE, index=False)


# ----------------------------------------------------------
# EN — 8️⃣ FINAL SUMMARY
# PT — 8️⃣ RELATÓRIO FINAL
# ----------------------------------------------------------

print("\n===== UPDATE SUMMARY =====")
print(f"Total records in base: {len(df_base)}")
print(f"Records updated to PDF = Yes: {updated_count}")
print(f"📄 Updated dataset saved at: {OUTPUT_FILE}")


📂 Files validated successfully.
📊 Base dataset size: 2944
📊 Update dataset size: 443

===== UPDATE SUMMARY =====
Total records in base: 2944
Records updated to PDF = Yes: 22
📄 Updated dataset saved at: /content/drive/MyDrive/Tese/ArticleSLR/data8.pdf.csv


In [ ]:
# ==========================================================
# EN — STEP 17: GENERATE A DATA WITH ARTICLES WITHOUT PDF
# PT — ETAPA 17: GERAR UMA BASE SOMENTE COM ARTIGOS SEM PDF
# ==========================================================


# ----------------------------------------------------------
# EN — 1️⃣ IMPORT REQUIRED LIBRARIES
# PT — 1️⃣ IMPORTAR BIBLIOTECAS NECESSÁRIAS
# ----------------------------------------------------------

import pandas as pd
from pathlib import Path


# ----------------------------------------------------------
# EN — 2️⃣ DEFINE PROJECT PATHS
# PT — 2️⃣ DEFINIR CAMINHOS DO PROJETO
# ----------------------------------------------------------

PROJECT_ROOT = Path("/content/drive/MyDrive/Tese/ArticleSLR")

INPUT_FILE = PROJECT_ROOT / "data8.pdf.csv"
OUTPUT_FILE = PROJECT_ROOT / "data0.nopdf2.csv"


# ----------------------------------------------------------
# EN — 3️⃣ VALIDATE INPUT FILE EXISTENCE
# PT — 3️⃣ VALIDAR EXISTÊNCIA DO ARQUIVO DE ENTRADA
# ----------------------------------------------------------

if not INPUT_FILE.exists():
    raise FileNotFoundError(f"Input file not found: {INPUT_FILE}")

print(f"📂 Input file confirmed: {INPUT_FILE}")


# ----------------------------------------------------------
# EN — 4️⃣ LOAD DATASET
# PT — 4️⃣ CARREGAR DATASET
# ----------------------------------------------------------

df = pd.read_csv(INPUT_FILE)

print(f"📊 Total articles loaded: {len(df)}")


# ----------------------------------------------------------
# EN — 5️⃣ VALIDATE REQUIRED COLUMN
# PT — 5️⃣ VALIDAR COLUNA OBRIGATÓRIA
# ----------------------------------------------------------

if "PDF" not in df.columns:
    raise ValueError("Required column 'PDF' not found in dataset.")


# ----------------------------------------------------------
# EN — 6️⃣ FILTER ARTICLES WITHOUT PDF
# PT — 6️⃣ FILTRAR ARTIGOS QUE NÃO POSSUEM PDF
# ----------------------------------------------------------

not_found_df = df[df["PDF"] == "No"]

total_missing = len(not_found_df)

print(f"📉 Articles without PDF: {total_missing}")


# ----------------------------------------------------------
# EN — 7️⃣ SAVE FILTERED DATASET
# PT — 7️⃣ SALVAR DATASET FILTRADO
# ----------------------------------------------------------

OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)

not_found_df.to_csv(OUTPUT_FILE, index=False)

print(f"💾 Output file saved at: {OUTPUT_FILE}")


# ----------------------------------------------------------
# EN — 8️⃣ FINAL SUMMARY
# PT — 8️⃣ RELATÓRIO FINAL
# ----------------------------------------------------------

recovery_rate = ((len(df) - total_missing) / len(df)) * 100 if len(df) > 0 else 0

print("\n===== SUMMARY =====")
print(f"Total articles: {len(df)}")
print(f"Articles with PDF: {len(df) - total_missing}")
print(f"Articles without PDF: {total_missing}")
print(f"Current recovery rate: {recovery_rate:.2f}%")

📂 Input file confirmed: /content/drive/MyDrive/Tese/ArticleSLR/data8.pdf.csv
📊 Total articles loaded: 2944
📉 Articles without PDF: 421
💾 Output file saved at: /content/drive/MyDrive/Tese/ArticleSLR/data0.nopdf2.csv

===== SUMMARY =====
Total articles: 2944
Articles with PDF: 2523
Articles without PDF: 421
Current recovery rate: 85.70%


In [ ]:
#==========================================================
# EN — STEP 18: EXPORT ARTICLES WITHOUT PDF TO ZOTERO (BIBTEX)
# PT — ETAPA 18: EXPORTAR ARTIGOS SEM PDF PARA O ZOTERO (BIBTEX)
#==========================================================

# ----------------------------------------------------------
# EN — 1️⃣ IMPORT REQUIRED LIBRARIES
# PT — 1️⃣ IMPORTAR BIBLIOTECAS NECESSÁRIAS
# ----------------------------------------------------------

import pandas as pd
from pathlib import Path


# ----------------------------------------------------------
# EN — 2️⃣ DEFINE PROJECT PATHS
# PT — 2️⃣ DEFINIR CAMINHOS DO PROJETO
# ----------------------------------------------------------

PROJECT_ROOT = Path("/content/drive/MyDrive/Tese/ArticleSLR")

INPUT_FILE = PROJECT_ROOT / "data0.nopdf2.csv"
OUTPUT_FILE = PROJECT_ROOT / "data0.nopdf2.bib"


# ----------------------------------------------------------
# EN — 3️⃣ VALIDATE INPUT FILE EXISTENCE
# PT — 3️⃣ VALIDAR EXISTÊNCIA DO ARQUIVO DE ENTRADA
# ----------------------------------------------------------

if not INPUT_FILE.exists():
    raise FileNotFoundError(f"❌ Input file not found: {INPUT_FILE}")

print(f"📂 Input file confirmed: {INPUT_FILE}")


# ----------------------------------------------------------
# EN — 4️⃣ LOAD DATASET
# PT — 4️⃣ CARREGAR DATASET
# ----------------------------------------------------------

df = pd.read_csv(INPUT_FILE)

print(f"📊 Total articles loaded: {len(df)}")


# ----------------------------------------------------------
# EN — 5️⃣ VALIDATE REQUIRED COLUMNS
# PT — 5️⃣ VALIDAR COLUNAS OBRIGATÓRIAS
# ----------------------------------------------------------

required_cols = ["Title", "DOI"]

for col in required_cols:
    if col not in df.columns:
        raise ValueError(f"❌ Required column '{col}' not found in dataset.")


# ----------------------------------------------------------
# EN — 6️⃣ CLEAN TEXT FOR BIBTEX SAFETY
# PT — 6️⃣ LIMPEZA DE TEXTO PARA SEGURANÇA BIBTEX
# ----------------------------------------------------------

def sanitize_bibtex(text):
    if pd.isna(text):
        return ""
    text = str(text)
    text = text.replace("{", "").replace("}", "")
    return text.strip()


# ----------------------------------------------------------
# EN — 7️⃣ FUNCTION TO GENERATE BIBTEX ENTRY
# PT — 7️⃣ FUNÇÃO PARA GERAR ENTRADA BIBTEX
# ----------------------------------------------------------

def to_bibtex_entry(index, row):

    entry_key = f"entry{index}"

    title = sanitize_bibtex(row["Title"])
    doi = sanitize_bibtex(row["DOI"])

    entry = f"@article{{{entry_key},\n"
    entry += f"  title = {{{title}}},\n"

    if "Author" in df.columns and pd.notna(row.get("Author")):
        entry += f"  author = {{{sanitize_bibtex(row['Author'])}}},\n"

    if "Year" in df.columns and pd.notna(row.get("Year")):
        try:
            entry += f"  year = {{{int(row['Year'])}}},\n"
        except:
            entry += f"  year = {{{sanitize_bibtex(row['Year'])}}},\n"

    if "Journal" in df.columns and pd.notna(row.get("Journal")):
        entry += f"  journal = {{{sanitize_bibtex(row['Journal'])}}},\n"

    entry += f"  doi = {{{doi}}}\n"
    entry += "}\n\n"

    return entry


# ----------------------------------------------------------
# EN — 8️⃣ GENERATE BIBTEX FILE
# PT — 8️⃣ GERAR ARQUIVO BIBTEX
# ----------------------------------------------------------

bib_entries = []

for idx, row in df.iterrows():

    if pd.notna(row["DOI"]) and str(row["DOI"]).strip() != "":
        bib_entries.append(to_bibtex_entry(idx, row))


with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    f.writelines(bib_entries)


# ----------------------------------------------------------
# EN — 9️⃣ FINAL SUMMARY
# PT — 9️⃣ RELATÓRIO FINAL
# ----------------------------------------------------------

print("\n===== EXPORT SUMMARY =====")
print(f"Total entries exported: {len(bib_entries)}")
print(f"BibTeX file saved at: {OUTPUT_FILE}")


📂 Input file confirmed: /content/drive/MyDrive/Tese/ArticleSLR/data0.nopdf2.csv
📊 Total articles loaded: 421

===== EXPORT SUMMARY =====
Total entries exported: 421
BibTeX file saved at: /content/drive/MyDrive/Tese/ArticleSLR/data0.nopdf2.bib


In [ ]:
# ==========================================================
# EN — STEP 19: CLEAN ZOTERO PDF NAMES AND SAVE TO MAIN FOLDER
# PT — ETAPA 19: LIMPAR NOMES DOS PDFs DO ZOTERO E SALVAR NA PASTA PRINCIPAL
# ==========================================================


# ----------------------------------------------------------
# EN — 1️⃣ IMPORT REQUIRED LIBRARIES
# PT — 1️⃣ IMPORTAR BIBLIOTECAS NECESSÁRIAS
# ----------------------------------------------------------

import shutil
import re
from pathlib import Path


# ----------------------------------------------------------
# EN — 2️⃣ DEFINE PROJECT PATHS
# PT — 2️⃣ DEFINIR CAMINHOS DO PROJETO
# ----------------------------------------------------------

BASE_DIR = Path("/content/drive/MyDrive/Tese/ArticleSLR/PDFs_Zotero")
SOURCE_DIR = BASE_DIR / "pdf_zotero"
TARGET_DIR = BASE_DIR

TARGET_DIR.mkdir(parents=True, exist_ok=True)

if not SOURCE_DIR.exists():
    raise FileNotFoundError(f"❌ Source folder not found: {SOURCE_DIR}")


# ----------------------------------------------------------
# EN — 3️⃣ HELPER FUNCTIONS
# PT — 3️⃣ FUNÇÕES AUXILIARES
# ----------------------------------------------------------

def remove_year_prefix(filename):
    """
    EN — Remove leading year pattern like '2025 - '
    PT — Remove padrão inicial de ano como '2025 - '
    """
    return re.sub(r"^\d{4}\s*-\s*", "", filename)


def clean_filename(text):
    """
    EN — Remove invalid characters and excessive spaces
    PT — Remove caracteres inválidos e espaços excessivos
    """
    text = re.sub(r"[\\/*?\"<>|:]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text[:200]


# ----------------------------------------------------------
# EN — 4️⃣ PROCESS AND COPY FILES
# PT — 4️⃣ PROCESSAR E COPIAR ARQUIVOS
# ----------------------------------------------------------

copied = 0
skipped = 0
errors = 0

for pdf_file in SOURCE_DIR.glob("*.pdf"):

    try:
        original_name = pdf_file.name

        # EN — Remove year prefix
        # PT — Remover prefixo do ano
        new_name = remove_year_prefix(original_name)

        # EN — Clean invalid characters
        # PT — Limpar caracteres inválidos
        new_name = clean_filename(new_name)

        destination_path = TARGET_DIR / new_name

        # EN — Avoid overwriting existing files
        # PT — Evitar sobrescrever arquivos existentes
        if destination_path.exists():
            print(f"⚠️ Already exists, skipping: {new_name}")
            skipped += 1
            continue

        shutil.copy2(pdf_file, destination_path)
        copied += 1

    except Exception as e:
        print(f"❌ Error processing {pdf_file.name}: {e}")
        errors += 1


# ----------------------------------------------------------
# EN — 5️⃣ FINAL SUMMARY
# PT — 5️⃣ RELATÓRIO FINAL
# ----------------------------------------------------------

print("\n===== RENAMING SUMMARY =====")
print(f"PDFs processed successfully: {copied}")
print(f"Skipped (already existed): {skipped}")
print(f"Errors: {errors}")
print(f"📂 Files saved to: {TARGET_DIR}")



===== RENAMING SUMMARY =====
PDFs processed successfully: 70
Skipped (already existed): 0
Errors: 0
📂 Files saved to: /content/drive/MyDrive/Tese/ArticleSLR/PDFs_Zotero


In [ ]:
# ==========================================================
# EN — STEP 20: UPDATE data8 USING ZOTERO METADATA (FINAL FIX)
# PT — ETAPA 20: ATUALIZAR data8 USANDO METADADOS DO ZOTERO
# ==========================================================


# ----------------------------------------------------------
# 1️⃣ IMPORT LIBRARIES
# ----------------------------------------------------------

import pandas as pd
import re
from pathlib import Path
from difflib import SequenceMatcher


# ----------------------------------------------------------
# 2️⃣ DEFINE PATHS
# ----------------------------------------------------------

PROJECT_ROOT = Path("/content/drive/MyDrive/Tese/ArticleSLR")

DATA8_FILE = PROJECT_ROOT / "data8.pdf.csv"
ZOTERO_METADATA = PROJECT_ROOT / "PDFs_Zotero" / "metadados_pdf_zotero.csv"
OUTPUT_FILE = PROJECT_ROOT / "data9.pdf.csv"


# ----------------------------------------------------------
# 3️⃣ LOAD DATA
# ----------------------------------------------------------

df_data8 = pd.read_csv(DATA8_FILE)
df_zotero = pd.read_csv(ZOTERO_METADATA)

print("📂 Files loaded successfully.")
print(f"📊 data8 articles: {len(df_data8)}")
print(f"📊 Zotero metadata articles: {len(df_zotero)}")


# ----------------------------------------------------------
# 4️⃣ VALIDATE REQUIRED COLUMNS
# ----------------------------------------------------------

required_columns = ["Title", "DOI", "Publication Year"]

for col in required_columns:
    if col not in df_zotero.columns:
        raise ValueError(f"Column '{col}' not found in Zotero metadata.")

print("✅ Required columns confirmed in Zotero metadata.")


# ----------------------------------------------------------
# 5️⃣ NORMALIZATION FUNCTIONS
# ----------------------------------------------------------

def normalize_text(text):
    text = str(text).lower()
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def normalize_doi(doi):
    if pd.isna(doi):
        return ""
    doi = str(doi).lower().strip()
    doi = doi.replace("https://doi.org/", "")
    doi = doi.replace("http://doi.org/", "")
    return doi


# ----------------------------------------------------------
# 6️⃣ PREPARE ZOTERO DATA
# ----------------------------------------------------------

df_zotero["DOI_norm"] = df_zotero["DOI"].apply(normalize_doi)
df_zotero["Title_norm"] = df_zotero["Title"].apply(normalize_text)
df_zotero["Year_norm"] = df_zotero["Publication Year"].astype(str).str.extract(r'(\d{4})')

zotero_doi_set = set(df_zotero["DOI_norm"].dropna())


# ----------------------------------------------------------
# 7️⃣ UPDATE LOGIC
# ----------------------------------------------------------

updated = 0
matched_by_doi = 0
matched_by_title_year = 0
matched_by_similarity = 0

for idx, row in df_data8.iterrows():

    doi_norm = normalize_doi(row.get("DOI", ""))
    title_norm = normalize_text(row.get("Title", ""))
    year_norm = str(row.get("Year", "")).strip()

    match_found = False

    # 1️⃣ MATCH BY DOI (MOST RELIABLE)
    if doi_norm and doi_norm in zotero_doi_set:
        match_found = True
        matched_by_doi += 1

    # 2️⃣ MATCH BY TITLE + YEAR
    if not match_found:
        possible = df_zotero[
            (df_zotero["Title_norm"] == title_norm) &
            (df_zotero["Year_norm"] == year_norm)
        ]
        if len(possible) > 0:
            match_found = True
            matched_by_title_year += 1

    # 3️⃣ MATCH BY HIGH TITLE SIMILARITY
    if not match_found:
        for _, zot_row in df_zotero.iterrows():
            score = SequenceMatcher(
                None,
                title_norm,
                zot_row["Title_norm"]
            ).ratio()
            if score >= 0.90:
                match_found = True
                matched_by_similarity += 1
                break

    if match_found:
        if df_data8.at[idx, "PDF"] != "Yes":
            df_data8.at[idx, "PDF"] = "Yes"
            updated += 1


# ----------------------------------------------------------
# 8️⃣ SAVE UPDATED DATASET
# ----------------------------------------------------------

df_data8.to_csv(OUTPUT_FILE, index=False)


# ----------------------------------------------------------
# 9️⃣ FINAL REPORT
# ----------------------------------------------------------

print("\n===== UPDATE SUMMARY =====")
print(f"Total articles: {len(df_data8)}")
print(f"Updated to PDF = Yes: {updated}")
print(f"Matched by DOI: {matched_by_doi}")
print(f"Matched by Title+Year: {matched_by_title_year}")
print(f"Matched by Similarity: {matched_by_similarity}")
print(f"📄 New dataset saved at: {OUTPUT_FILE}")


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/Tese/ArticleSLR/data8.pdf.csv'

In [ ]:
# ==========================================================
# EN — STEP 21: GENERATE A DATA WITH ARTICLES WITHOUT PDF
# PT — ETAPA 21: GERAR UMA BASE SOMENTE COM ARTIGOS SEM PDF
# ==========================================================


# ----------------------------------------------------------
# EN — 1️⃣ IMPORT REQUIRED LIBRARIES
# PT — 1️⃣ IMPORTAR BIBLIOTECAS NECESSÁRIAS
# ----------------------------------------------------------

import pandas as pd
from pathlib import Path


# ----------------------------------------------------------
# EN — 2️⃣ DEFINE PROJECT PATHS
# PT — 2️⃣ DEFINIR CAMINHOS DO PROJETO
# ----------------------------------------------------------

PROJECT_ROOT = Path("/content/drive/MyDrive/Tese/ArticleSLR")

INPUT_FILE = PROJECT_ROOT / "data9.pdf.csv"
OUTPUT_FILE = PROJECT_ROOT / "data0.nopdf3.csv"


# ----------------------------------------------------------
# EN — 3️⃣ VALIDATE INPUT FILE EXISTENCE
# PT — 3️⃣ VALIDAR EXISTÊNCIA DO ARQUIVO DE ENTRADA
# ----------------------------------------------------------

if not INPUT_FILE.exists():
    raise FileNotFoundError(f"Input file not found: {INPUT_FILE}")

print(f"📂 Input file confirmed: {INPUT_FILE}")


# ----------------------------------------------------------
# EN — 4️⃣ LOAD DATASET
# PT — 4️⃣ CARREGAR DATASET
# ----------------------------------------------------------

df = pd.read_csv(INPUT_FILE)

print(f"📊 Total articles loaded: {len(df)}")


# ----------------------------------------------------------
# EN — 5️⃣ VALIDATE REQUIRED COLUMN
# PT — 5️⃣ VALIDAR COLUNA OBRIGATÓRIA
# ----------------------------------------------------------

if "PDF" not in df.columns:
    raise ValueError("Required column 'PDF' not found in dataset.")


# ----------------------------------------------------------
# EN — 6️⃣ FILTER ARTICLES WITHOUT PDF
# PT — 6️⃣ FILTRAR ARTIGOS QUE NÃO POSSUEM PDF
# ----------------------------------------------------------

not_found_df = df[df["PDF"] == "No"]

total_missing = len(not_found_df)

print(f"📉 Articles without PDF: {total_missing}")


# ----------------------------------------------------------
# EN — 7️⃣ SAVE FILTERED DATASET
# PT — 7️⃣ SALVAR DATASET FILTRADO
# ----------------------------------------------------------

OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)

not_found_df.to_csv(OUTPUT_FILE, index=False)

print(f"💾 Output file saved at: {OUTPUT_FILE}")


# ----------------------------------------------------------
# EN — 8️⃣ FINAL SUMMARY
# PT — 8️⃣ RELATÓRIO FINAL
# ----------------------------------------------------------

recovery_rate = ((len(df) - total_missing) / len(df)) * 100 if len(df) > 0 else 0

print("\n===== SUMMARY =====")
print(f"Total articles: {len(df)}")
print(f"Articles with PDF: {len(df) - total_missing}")
print(f"Articles without PDF: {total_missing}")
print(f"Current recovery rate: {recovery_rate:.2f}%")

📂 Input file confirmed: /content/drive/MyDrive/Tese/ArticleSLR/data9.pdf.csv
📊 Total articles loaded: 2944
📉 Articles without PDF: 351
💾 Output file saved at: /content/drive/MyDrive/Tese/ArticleSLR/data0.nopdf3.csv

===== SUMMARY =====
Total articles: 2944
Articles with PDF: 2593
Articles without PDF: 351
Current recovery rate: 88.08%


In [ ]:
# ==========================================================
# EN — STEP 22 : CLEAN MANUAL PDF NAMES
# PT — ETAPA 22 : LIMPAR NOMES DOS PDFs LOCALIZADOS MANUALMENTE
# ==========================================================

import shutil
import re
from pathlib import Path

# ----------------------------------------------------------
# EN — 1️⃣ DEFINE PROJECT PATHS
# PT — 1️⃣ DEFINIR CAMINHOS DO PROJETO
# ----------------------------------------------------------

PROJECT_ROOT = Path("/content/drive/MyDrive/Tese/ArticleSLR")

BASE_DIR = PROJECT_ROOT / "PDFs_manually"
SOURCE_DIR = BASE_DIR / "manually"
TARGET_DIR = BASE_DIR

if not SOURCE_DIR.exists():
    raise FileNotFoundError(f"❌ Source folder not found: {SOURCE_DIR}")

print("📂 Folder structure validated.")


# ----------------------------------------------------------
# EN — 2️⃣ HELPER FUNCTIONS
# PT — 2️⃣ FUNÇÕES AUXILIARES
# ----------------------------------------------------------

def remove_author_and_year(filename):
    return re.sub(r"^.*?\-\s*\d{4}\s*-\s*", "", filename)

def clean_filename(text):
    text = re.sub(r"[\\/*?\"<>|:]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text[:200]


# ----------------------------------------------------------
# EN — 3️⃣ BUILD QUICK SIZE INDEX (FASTER DUPLICATE CHECK)
# PT — 3️⃣ CRIAR ÍNDICE DE TAMANHO (DETECÇÃO RÁPIDA)
# ----------------------------------------------------------

size_index = {}
for existing_file in TARGET_DIR.glob("*.pdf"):
    size_index.setdefault(existing_file.stat().st_size, []).append(existing_file.name)


# ----------------------------------------------------------
# EN — 4️⃣ PROCESS FILES
# PT — 4️⃣ PROCESSAR ARQUIVOS
# ----------------------------------------------------------

processed = 0
skipped = 0
errors = 0

for pdf_file in SOURCE_DIR.glob("*.pdf"):

    try:
        original_name = pdf_file.name

        new_name = remove_author_and_year(original_name)
        new_name = clean_filename(new_name)

        destination_path = TARGET_DIR / new_name

        # --------------------------------------------------
        # EN — Case 1: Already correctly named in target
        # PT — Caso 1: Já existe com nome correto
        # --------------------------------------------------
        if destination_path.exists():
            skipped += 1
            continue

        # --------------------------------------------------
        # EN — Case 2: Same file size already exists
        # PT — Caso 2: Mesmo tamanho já existe
        # --------------------------------------------------
        file_size = pdf_file.stat().st_size

        if file_size in size_index:
            skipped += 1
            continue

        # --------------------------------------------------
        # EN — Copy only if really necessary
        # PT — Copiar apenas se necessário
        # --------------------------------------------------
        shutil.copy2(pdf_file, destination_path)
        processed += 1

        # Update size index dynamically
        size_index.setdefault(file_size, []).append(new_name)

    except Exception as e:
        print(f"❌ Error processing {pdf_file.name}: {e}")
        errors += 1


# ----------------------------------------------------------
# EN — 5️⃣ FINAL SUMMARY
# PT — 5️⃣ RELATÓRIO FINAL
# ----------------------------------------------------------

print("\n===== MANUAL PDF RENAMING SUMMARY =====")
print(f"PDFs processed successfully: {processed}")
print(f"Skipped (already existed or duplicate size): {skipped}")
print(f"Errors: {errors}")
print(f"📂 Files saved to: {TARGET_DIR}")

📂 Folder structure validated.

===== MANUAL PDF RENAMING SUMMARY =====
PDFs processed successfully: 8
Skipped (already existed or duplicate size): 55
Errors: 0
📂 Files saved to: /content/drive/MyDrive/Tese/ArticleSLR/PDFs_manually


In [ ]:
# ==========================================================
# EN — STEP 23: UPDATE data9 USING METADATA FROM MANUALLY SAVED ARTICLES
# PT — ETAPA 23: ATUALIZAR data9 USANDO METADADOS DOS ARTIGOS SALVOS MANUALMENTE
# ==========================================================


# ----------------------------------------------------------
# 1️⃣ IMPORT LIBRARIES
# ----------------------------------------------------------

import pandas as pd
import re
from pathlib import Path
from difflib import SequenceMatcher


# ----------------------------------------------------------
# 2️⃣ DEFINE PATHS
# ----------------------------------------------------------

PROJECT_ROOT = Path("/content/drive/MyDrive/Tese/ArticleSLR")

DATA8_FILE = PROJECT_ROOT / "data9.pdf.csv"
ZOTERO_METADATA = PROJECT_ROOT / "PDFs_manually" / "metadados_pdf_manually.csv"
OUTPUT_FILE = PROJECT_ROOT / "data10.pdf.csv"


# ----------------------------------------------------------
# 3️⃣ LOAD DATA
# ----------------------------------------------------------

df_data8 = pd.read_csv(DATA8_FILE)
df_zotero = pd.read_csv(ZOTERO_METADATA)

print("📂 Files loaded successfully.")
print(f"📊 data8 articles: {len(df_data8)}")
print(f"📊 Zotero metadata articles: {len(df_zotero)}")


# ----------------------------------------------------------
# 4️⃣ VALIDATE REQUIRED COLUMNS
# ----------------------------------------------------------

required_columns = ["Title", "DOI", "Publication Year"]

for col in required_columns:
    if col not in df_zotero.columns:
        raise ValueError(f"Column '{col}' not found in Zotero metadata.")

print("✅ Required columns confirmed in Zotero metadata.")


# ----------------------------------------------------------
# 5️⃣ NORMALIZATION FUNCTIONS
# ----------------------------------------------------------

def normalize_text(text):
    text = str(text).lower()
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def normalize_doi(doi):
    if pd.isna(doi):
        return ""
    doi = str(doi).lower().strip()
    doi = doi.replace("https://doi.org/", "")
    doi = doi.replace("http://doi.org/", "")
    return doi


# ----------------------------------------------------------
# 6️⃣ PREPARE ZOTERO DATA
# ----------------------------------------------------------

df_zotero["DOI_norm"] = df_zotero["DOI"].apply(normalize_doi)
df_zotero["Title_norm"] = df_zotero["Title"].apply(normalize_text)
df_zotero["Year_norm"] = df_zotero["Publication Year"].astype(str).str.extract(r'(\d{4})')

zotero_doi_set = set(df_zotero["DOI_norm"].dropna())


# ----------------------------------------------------------
# 7️⃣ UPDATE LOGIC
# ----------------------------------------------------------

updated = 0
matched_by_doi = 0
matched_by_title_year = 0
matched_by_similarity = 0

for idx, row in df_data8.iterrows():

    doi_norm = normalize_doi(row.get("DOI", ""))
    title_norm = normalize_text(row.get("Title", ""))
    year_norm = str(row.get("Year", "")).strip()

    match_found = False

    # 1️⃣ MATCH BY DOI (MOST RELIABLE)
    if doi_norm and doi_norm in zotero_doi_set:
        match_found = True
        matched_by_doi += 1

    # 2️⃣ MATCH BY TITLE + YEAR
    if not match_found:
        possible = df_zotero[
            (df_zotero["Title_norm"] == title_norm) &
            (df_zotero["Year_norm"] == year_norm)
        ]
        if len(possible) > 0:
            match_found = True
            matched_by_title_year += 1

    # 3️⃣ MATCH BY HIGH TITLE SIMILARITY
    if not match_found:
        for _, zot_row in df_zotero.iterrows():
            score = SequenceMatcher(
                None,
                title_norm,
                zot_row["Title_norm"]
            ).ratio()
            if score >= 0.90:
                match_found = True
                matched_by_similarity += 1
                break

    if match_found:
        if df_data8.at[idx, "PDF"] != "Yes":
            df_data8.at[idx, "PDF"] = "Yes"
            updated += 1


# ----------------------------------------------------------
# 8️⃣ SAVE UPDATED DATASET
# ----------------------------------------------------------

df_data8.to_csv(OUTPUT_FILE, index=False)


# ----------------------------------------------------------
# 9️⃣ FINAL REPORT
# ----------------------------------------------------------

print("\n===== UPDATE SUMMARY =====")
print(f"Total articles: {len(df_data8)}")
print(f"Updated to PDF = Yes: {updated}")
print(f"Matched by DOI: {matched_by_doi}")
print(f"Matched by Title+Year: {matched_by_title_year}")
print(f"Matched by Similarity: {matched_by_similarity}")
print(f"📄 New dataset saved at: {OUTPUT_FILE}")


📂 Files loaded successfully.
📊 data8 articles: 2944
📊 Zotero metadata articles: 337
✅ Required columns confirmed in Zotero metadata.

===== UPDATE SUMMARY =====
Total articles: 2944
Updated to PDF = Yes: 335
Matched by DOI: 334
Matched by Title+Year: 6
Matched by Similarity: 6
📄 New dataset saved at: /content/drive/MyDrive/Tese/ArticleSLR/data10.pdf.csv


In [ ]:
# ==========================================================
# EN — STEP 24: GENERATE A DATA WITH ARTICLES WITHOUT PDF
# PT — ETAPA 24: GERAR UMA BASE SOMENTE COM ARTIGOS SEM PDF
# ==========================================================


# ----------------------------------------------------------
# EN — 1️⃣ IMPORT REQUIRED LIBRARIES
# PT — 1️⃣ IMPORTAR BIBLIOTECAS NECESSÁRIAS
# ----------------------------------------------------------

import pandas as pd
from pathlib import Path


# ----------------------------------------------------------
# EN — 2️⃣ DEFINE PROJECT PATHS
# PT — 2️⃣ DEFINIR CAMINHOS DO PROJETO
# ----------------------------------------------------------

PROJECT_ROOT = Path("/content/drive/MyDrive/Tese/ArticleSLR")

INPUT_FILE = PROJECT_ROOT / "data10.pdf.csv"
OUTPUT_FILE = PROJECT_ROOT / "data0.nopdf4.csv"


# ----------------------------------------------------------
# EN — 3️⃣ VALIDATE INPUT FILE EXISTENCE
# PT — 3️⃣ VALIDAR EXISTÊNCIA DO ARQUIVO DE ENTRADA
# ----------------------------------------------------------

if not INPUT_FILE.exists():
    raise FileNotFoundError(f"Input file not found: {INPUT_FILE}")

print(f"📂 Input file confirmed: {INPUT_FILE}")


# ----------------------------------------------------------
# EN — 4️⃣ LOAD DATASET
# PT — 4️⃣ CARREGAR DATASET
# ----------------------------------------------------------

df = pd.read_csv(INPUT_FILE)

print(f"📊 Total articles loaded: {len(df)}")


# ----------------------------------------------------------
# EN — 5️⃣ VALIDATE REQUIRED COLUMN
# PT — 5️⃣ VALIDAR COLUNA OBRIGATÓRIA
# ----------------------------------------------------------

if "PDF" not in df.columns:
    raise ValueError("Required column 'PDF' not found in dataset.")


# ----------------------------------------------------------
# EN — 6️⃣ FILTER ARTICLES WITHOUT PDF
# PT — 6️⃣ FILTRAR ARTIGOS QUE NÃO POSSUEM PDF
# ----------------------------------------------------------

not_found_df = df[df["PDF"] == "No"]

total_missing = len(not_found_df)

print(f"📉 Articles without PDF: {total_missing}")


# ----------------------------------------------------------
# EN — 7️⃣ SAVE FILTERED DATASET
# PT — 7️⃣ SALVAR DATASET FILTRADO
# ----------------------------------------------------------

OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)

not_found_df.to_csv(OUTPUT_FILE, index=False)

print(f"💾 Output file saved at: {OUTPUT_FILE}")


# ----------------------------------------------------------
# EN — 8️⃣ FINAL SUMMARY
# PT — 8️⃣ RELATÓRIO FINAL
# ----------------------------------------------------------

recovery_rate = ((len(df) - total_missing) / len(df)) * 100 if len(df) > 0 else 0

print("\n===== SUMMARY =====")
print(f"Total articles: {len(df)}")
print(f"Articles with PDF: {len(df) - total_missing}")
print(f"Articles without PDF: {total_missing}")
print(f"Current recovery rate: {recovery_rate:.2f}%")

📂 Input file confirmed: /content/drive/MyDrive/Tese/ArticleSLR/data10.pdf.csv
📊 Total articles loaded: 2944
📉 Articles without PDF: 16
💾 Output file saved at: /content/drive/MyDrive/Tese/ArticleSLR/data0.nopdf4.csv

===== SUMMARY =====
Total articles: 2944
Articles with PDF: 2928
Articles without PDF: 16
Current recovery rate: 99.46%


In [ ]:
# ==========================================================
# EN — STEP 25: UPDATE data10 USING METADATA EXTRACTED FROM EMAIL PDFs
# PT — ETAPA 25: ATUALIZAR data10 USANDO METADADOS EXTRAÍDOS DOS PDFs RECEBIDOS
# ==========================================================


# ----------------------------------------------------------
# 0️⃣ INSTALL REQUIRED LIBRARY
# ----------------------------------------------------------

!pip install pymupdf


# ----------------------------------------------------------
# 1️⃣ IMPORT LIBRARIES
# ----------------------------------------------------------

import pandas as pd
import re
from pathlib import Path
from difflib import SequenceMatcher
import fitz  # PyMuPDF


# ----------------------------------------------------------
# 2️⃣ DEFINE PATHS
# ----------------------------------------------------------

PROJECT_ROOT = Path("/content/drive/MyDrive/Tese/ArticleSLR")

DATA10_FILE = PROJECT_ROOT / "data10.pdf.csv"
EMAIL_PDF_FOLDER = PROJECT_ROOT / "PDFs_e-mail"
OUTPUT_FILE = PROJECT_ROOT / "data11.pdf.csv"


# ----------------------------------------------------------
# 3️⃣ LOAD DATA
# ----------------------------------------------------------

df_data10 = pd.read_csv(DATA10_FILE)

print("📂 data10 loaded successfully.")
print(f"📊 data10 articles: {len(df_data10)}")


# ----------------------------------------------------------
# 4️⃣ NORMALIZATION FUNCTIONS
# ----------------------------------------------------------

def normalize_text(text):
    text = str(text).lower()
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def normalize_doi(doi):
    if pd.isna(doi):
        return ""
    doi = str(doi).lower().strip()
    doi = doi.replace("https://doi.org/", "")
    doi = doi.replace("http://doi.org/", "")
    return doi


def extract_doi_from_text(text):
    doi_pattern = r"10\.\d{4,9}/[-._;()/:A-Z0-9]+"
    match = re.search(doi_pattern, text, re.I)
    if match:
        return normalize_doi(match.group(0))
    return ""


# ----------------------------------------------------------
# 5️⃣ EXTRACT METADATA FROM EMAIL PDFs
# ----------------------------------------------------------

pdf_files = list(EMAIL_PDF_FOLDER.glob("*.pdf"))
print(f"📩 PDFs found: {len(pdf_files)}")

pdf_metadata_list = []

for pdf_path in pdf_files:
    try:
        doc = fitz.open(pdf_path)
        metadata = doc.metadata

        title = metadata.get("title", "")
        authors = metadata.get("author", "")
        creation_date = metadata.get("creationDate", "")

        # Extract year from metadata
        year = ""
        if creation_date:
            year_match = re.search(r"(19|20)\d{2}", creation_date)
            if year_match:
                year = year_match.group(0)

        # Extract DOI from first page text
        first_page = doc[0].get_text()
        doi = extract_doi_from_text(first_page)

        # Fallback: if no metadata title, use first line of first page
        if not title:
            lines = first_page.split("\n")
            if lines:
                title = lines[0]

        pdf_metadata_list.append({
            "Title": title,
            "Authors": authors,
            "Year": year,
            "DOI": doi
        })

        doc.close()

    except Exception as e:
        print(f"⚠️ Error reading {pdf_path.name}: {e}")

df_email = pd.DataFrame(pdf_metadata_list)

print(f"📄 Extracted metadata from {len(df_email)} PDFs")


# ----------------------------------------------------------
# 6️⃣ NORMALIZE EMAIL METADATA
# ----------------------------------------------------------

df_email["Title_norm"] = df_email["Title"].apply(normalize_text)
df_email["DOI_norm"] = df_email["DOI"].apply(normalize_doi)
df_email["Year_norm"] = df_email["Year"].astype(str).str.extract(r'(\d{4})')

email_doi_set = set(df_email["DOI_norm"].dropna())


# ----------------------------------------------------------
# 7️⃣ UPDATE LOGIC
# ----------------------------------------------------------

updated = 0
matched_by_doi = 0
matched_by_title_year = 0
matched_by_similarity = 0

for idx, row in df_data10.iterrows():

    doi_norm = normalize_doi(row.get("DOI", ""))
    title_norm = normalize_text(row.get("Title", ""))
    year_norm = str(row.get("Year", "")).strip()

    match_found = False

    # 1️⃣ MATCH BY DOI
    if doi_norm and doi_norm in email_doi_set:
        match_found = True
        matched_by_doi += 1

    # 2️⃣ MATCH BY TITLE + YEAR
    if not match_found:
        possible = df_email[
            (df_email["Title_norm"] == title_norm) &
            (df_email["Year_norm"] == year_norm)
        ]
        if len(possible) > 0:
            match_found = True
            matched_by_title_year += 1

    # 3️⃣ MATCH BY HIGH TITLE SIMILARITY
    if not match_found:
        for _, email_row in df_email.iterrows():
            score = SequenceMatcher(
                None,
                title_norm,
                email_row["Title_norm"]
            ).ratio()
            if score >= 0.90:
                match_found = True
                matched_by_similarity += 1
                break

    # UPDATE FLAG
    if match_found:
        if df_data10.at[idx, "PDF"] != "Yes":
            df_data10.at[idx, "PDF"] = "Yes"
            updated += 1


# ----------------------------------------------------------
# 8️⃣ SAVE UPDATED DATASET
# ----------------------------------------------------------

df_data10.to_csv(OUTPUT_FILE, index=False)


# ----------------------------------------------------------
# 9️⃣ FINAL REPORT
# ----------------------------------------------------------

print("\n===== UPDATE SUMMARY =====")
print(f"Total articles: {len(df_data10)}")
print(f"Updated to PDF = Yes: {updated}")
print(f"Matched by DOI: {matched_by_doi}")
print(f"Matched by Title+Year: {matched_by_title_year}")
print(f"Matched by Similarity: {matched_by_similarity}")
print(f"📄 New dataset saved at: {OUTPUT_FILE}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 51.6 MB/s eta 0:00:00
📂 data10 loaded successfully.
📊 data10 articles: 2944
📩 PDFs found: 2
📄 Extracted metadata from 2 PDFs

===== UPDATE SUMMARY =====
Total articles: 2944
Updated to PDF = Yes: 2
Matched by DOI: 2
Matched by Title+Year: 0
Matched by Similarity: 0
📄 New dataset saved at: /content/drive/MyDrive/Tese/ArticleSLR/data11.pdf.csv


In [ ]:
# ==========================================================
# EN — STEP 24: GENERATE A DATA WITH ARTICLES WITHOUT PDF
# PT — ETAPA 24: GERAR UMA BASE SOMENTE COM ARTIGOS SEM PDF
# ==========================================================


# ----------------------------------------------------------
# EN — 1️⃣ IMPORT REQUIRED LIBRARIES
# PT — 1️⃣ IMPORTAR BIBLIOTECAS NECESSÁRIAS
# ----------------------------------------------------------

import pandas as pd
from pathlib import Path


# ----------------------------------------------------------
# EN — 2️⃣ DEFINE PROJECT PATHS
# PT — 2️⃣ DEFINIR CAMINHOS DO PROJETO
# ----------------------------------------------------------

PROJECT_ROOT = Path("/content/drive/MyDrive/Tese/ArticleSLR")

INPUT_FILE = PROJECT_ROOT / "data11.pdf.csv"
OUTPUT_FILE = PROJECT_ROOT / "data0.nopdf5.csv"


# ----------------------------------------------------------
# EN — 3️⃣ VALIDATE INPUT FILE EXISTENCE
# PT — 3️⃣ VALIDAR EXISTÊNCIA DO ARQUIVO DE ENTRADA
# ----------------------------------------------------------

if not INPUT_FILE.exists():
    raise FileNotFoundError(f"Input file not found: {INPUT_FILE}")

print(f"📂 Input file confirmed: {INPUT_FILE}")


# ----------------------------------------------------------
# EN — 4️⃣ LOAD DATASET
# PT — 4️⃣ CARREGAR DATASET
# ----------------------------------------------------------

df = pd.read_csv(INPUT_FILE)

print(f"📊 Total articles loaded: {len(df)}")


# ----------------------------------------------------------
# EN — 5️⃣ VALIDATE REQUIRED COLUMN
# PT — 5️⃣ VALIDAR COLUNA OBRIGATÓRIA
# ----------------------------------------------------------

if "PDF" not in df.columns:
    raise ValueError("Required column 'PDF' not found in dataset.")


# ----------------------------------------------------------
# EN — 6️⃣ FILTER ARTICLES WITHOUT PDF
# PT — 6️⃣ FILTRAR ARTIGOS QUE NÃO POSSUEM PDF
# ----------------------------------------------------------

not_found_df = df[df["PDF"] == "No"]

total_missing = len(not_found_df)

print(f"📉 Articles without PDF: {total_missing}")


# ----------------------------------------------------------
# EN — 7️⃣ SAVE FILTERED DATASET
# PT — 7️⃣ SALVAR DATASET FILTRADO
# ----------------------------------------------------------

OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)

not_found_df.to_csv(OUTPUT_FILE, index=False)

print(f"💾 Output file saved at: {OUTPUT_FILE}")


# ----------------------------------------------------------
# EN — 8️⃣ FINAL SUMMARY
# PT — 8️⃣ RELATÓRIO FINAL
# ----------------------------------------------------------

recovery_rate = ((len(df) - total_missing) / len(df)) * 100 if len(df) > 0 else 0

print("\n===== SUMMARY =====")
print(f"Total articles: {len(df)}")
print(f"Articles with PDF: {len(df) - total_missing}")
print(f"Articles without PDF: {total_missing}")
print(f"Current recovery rate: {recovery_rate:.2f}%")

📂 Input file confirmed: /content/drive/MyDrive/Tese/ArticleSLR/data11.pdf.csv
📊 Total articles loaded: 2944
📉 Articles without PDF: 14
💾 Output file saved at: /content/drive/MyDrive/Tese/ArticleSLR/data0.nopdf5.csv

===== SUMMARY =====
Total articles: 2944
Articles with PDF: 2930
Articles without PDF: 14
Current recovery rate: 99.52%


In [ ]:
#==========================================================
# EN — STEP 25: GENERATE Q1 CSV + COPY PDFs (2024–2026, Business/Management)
# PT — ETAPA 25: SELECIONAR ARTIGOS Q1 (2024–2026, Business/Management)
#==========================================================


# EN — 1️⃣ IMPORT REQUIRED LIBRARIES
# PT — 1️⃣ IMPORTAR BIBLIOTECAS NECESSÁRIAS
# ----------------------------------------------------------

import pandas as pd
import shutil
import re
from pathlib import Path


# ----------------------------------------------------------
# EN — 2️⃣ DEFINE PROJECT PATHS
# PT — 2️⃣ DEFINIR CAMINHOS DO PROJETO
# ----------------------------------------------------------

PROJECT_ROOT = Path("/content/drive/MyDrive/Tese/ArticleSLR")

INPUT_FILE = PROJECT_ROOT / "data10.pdf.csv"
PDF_SOURCE_DIR = PROJECT_ROOT / "PDFs"

OUTPUT_CSV = PROJECT_ROOT / "data0.Q1_24a26_BusinessManagement.csv"
PDF_TARGET_DIR = PROJECT_ROOT / "PDFs_Q1_24a26_BusinessManagement"

PDF_TARGET_DIR.mkdir(parents=True, exist_ok=True)


# ----------------------------------------------------------
# EN — 3️⃣ VALIDATE INPUT FILE
# PT — 3️⃣ VALIDAR ARQUIVO DE ENTRADA
# ----------------------------------------------------------

if not INPUT_FILE.exists():
    raise FileNotFoundError(f"Input file not found: {INPUT_FILE}")

print(f"📂 Input dataset confirmed: {INPUT_FILE}")


# ----------------------------------------------------------
# EN — 4️⃣ LOAD DATASET
# PT — 4️⃣ CARREGAR DATASET
# ----------------------------------------------------------

df = pd.read_csv(INPUT_FILE)

print(f"📊 Total articles in dataset: {len(df)}")


# ----------------------------------------------------------
# EN — 5️⃣ VALIDATE REQUIRED COLUMNS
# PT — 5️⃣ VALIDAR COLUNAS OBRIGATÓRIAS
# ----------------------------------------------------------

required_columns = ["Title", "PDF", "Impact", "Year", "Journal Area"]

for col in required_columns:
    if col not in df.columns:
        raise ValueError(f"Required column '{col}' not found in dataset.")


# ----------------------------------------------------------
# EN — 6️⃣ FILTER Q1 ARTICLES (2024–2026) + BUSINESS/MANAGEMENT
# PT — 6️⃣ FILTRAR Q1 (2024–2026) + BUSINESS/MANAGEMENT
# ----------------------------------------------------------

target_years = [2024, 2025, 2026]

filtered_df = df[
    (df["PDF"] == "Yes") &
    (df["Impact"] == "Q1") &
    (df["Year"].isin(target_years)) &
    (
        df["Journal Area"]
        .fillna("")
        .str.contains("Business|Management", case=False, regex=True)
    )
]

print(f"📈 Q1 (2024–2026) Business/Management with PDF: {len(filtered_df)}")


# ----------------------------------------------------------
# EN — 7️⃣ HELPER FUNCTION FOR SAFE FILENAMES
# PT — 7️⃣ FUNÇÃO AUXILIAR PARA NOMES SEGUROS
# ----------------------------------------------------------

def clean_filename(text):
    text = re.sub(r"[\\/*?\"<>|:]", "", str(text))
    text = re.sub(r"\s+", " ", text).strip()
    return text[:200]


# ----------------------------------------------------------
# EN — 8️⃣ COPY PDFs TO TARGET FOLDER
# PT — 8️⃣ COPIAR PDFs PARA A PASTA DESTINO
# ----------------------------------------------------------

copied = 0
missing_files = 0

for _, row in filtered_df.iterrows():

    title = clean_filename(row["Title"])
    source_pdf = PDF_SOURCE_DIR / f"{title}.pdf"
    destination_pdf = PDF_TARGET_DIR / f"{title}.pdf"

    if source_pdf.exists():
        shutil.copy2(source_pdf, destination_pdf)
        copied += 1
    else:
        missing_files += 1


# ----------------------------------------------------------
# EN — 9️⃣ SAVE FILTERED CSV
# PT — 9️⃣ SALVAR CSV FILTRADO
# ----------------------------------------------------------

filtered_df.to_csv(OUTPUT_CSV, index=False)


# ----------------------------------------------------------
# EN — 🔟 FINAL SUMMARY
# PT — 🔟 RELATÓRIO FINAL
# ----------------------------------------------------------

print("\n===== STEP 17 SUMMARY =====")
print(f"Total Q1 (2024–2026) Business/Management: {len(filtered_df)}")
print(f"PDFs successfully copied: {copied}")
print(f"PDF files missing: {missing_files}")
print(f"\n📂 Target folder: {PDF_TARGET_DIR}")
print(f"📄 Output CSV: {OUTPUT_CSV}")


📂 Input dataset confirmed: /content/drive/MyDrive/Tese/ArticleSLR/data10.pdf.csv
📊 Total articles in dataset: 2944
📈 Q1 (2024–2026) Business/Management with PDF: 304

===== STEP 17 SUMMARY =====
Total Q1 (2024–2026) Business/Management: 304
PDFs successfully copied: 232
PDF files missing: 72

📂 Target folder: /content/drive/MyDrive/Tese/ArticleSLR/PDFs_Q1_24a26_BusinessManagement
📄 Output CSV: /content/drive/MyDrive/Tese/ArticleSLR/data0.Q1_24a26_BusinessManagement.csv


In [ ]:
# ==========================================================
# EN — STEP 25: INTEGRATE METADATA (data11) + FULL-TEXT PDFs
# PT — ETAPA 25: INTEGRAR METADADOS (data11) + PDFs FULL-TEXT
# ==========================================================

# ----------------------------------------------------------
# 0️⃣ INSTALL LIBRARIES
# ----------------------------------------------------------
!pip install pymupdf pandas openpyxl tqdm --quiet

# ----------------------------------------------------------
# 1️⃣ IMPORTS
# ----------------------------------------------------------
import re
import fitz
import pandas as pd
from pathlib import Path
from tqdm import tqdm

# ----------------------------------------------------------
# 2️⃣ CONFIGURATION
# ----------------------------------------------------------
ROOT = Path("/content/drive/MyDrive/Tese/ArticleSLR")
METADATA_FILE = ROOT / "data11.pdf.csv"
PDF_FOLDER = ROOT / "PDFs"

OUTPUT_CSV = ROOT / "artigos_analise.csv"
OUTPUT_XLSX = ROOT / "artigos_analise.xlsx"
LOG_FILE = ROOT / "log_erros.csv"

# ----------------------------------------------------------
# 3️⃣ LOAD METADATA BASE
# ----------------------------------------------------------
df_meta = pd.read_csv(METADATA_FILE)

print("📊 Total artigos na base:", len(df_meta))

# Se existir coluna PDF, filtrar apenas os que têm PDF
if "PDF" in df_meta.columns:
    df_meta = df_meta[df_meta["PDF"] == "Yes"].copy()
    print("📄 Artigos com PDF disponível:", len(df_meta))

# ----------------------------------------------------------
# 4️⃣ NORMALIZATION FUNCTIONS
# ----------------------------------------------------------
def normalize(text):
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def normalize_doi(doi):
    if pd.isna(doi):
        return ""
    doi = str(doi).lower().strip()
    doi = doi.replace("https://doi.org/", "")
    return doi

def extract_doi(text):
    pattern = r"10\.\d{4,9}/[-._;()/:A-Z0-9]+"
    match = re.search(pattern, text, re.I)
    return normalize_doi(match.group(0)) if match else ""

# Criar índices para matching
df_meta["DOI_norm"] = df_meta["DOI"].apply(normalize_doi)
df_meta["Title_norm"] = df_meta["Title"].apply(normalize)

doi_index = dict(zip(df_meta["DOI_norm"], df_meta.index))
title_index = dict(zip(df_meta["Title_norm"], df_meta.index))

# ----------------------------------------------------------
# 5️⃣ EXTRAÇÃO CONCEITUAL
# ----------------------------------------------------------
def detect_methodology(text):
    t = text.lower()
    if "systematic review" in t:
        return "Review - Systematic"
    if "meta-analysis" in t:
        return "Review - Meta-analysis"
    if "experiment" in t:
        return "Empirical - Experiment"
    if "survey" in t:
        return "Empirical - Survey"
    if "case study" in t:
        return "Empirical - Case Study"
    return None

def detect_platform(text):
    platforms = ["google","facebook","twitter","youtube",
                 "instagram","tiktok","chatgpt","llm",
                 "search engine","recommendation algorithm"]
    found = [p for p in platforms if p in text.lower()]
    return ", ".join(found) if found else None

def detect_behavior(text):
    behaviors = ["search","decision","polarization",
                 "engagement","consumption"]
    found = [b for b in behaviors if b in text.lower()]
    return ", ".join(found) if found else None

def extract_section(text, keyword):
    pattern = rf"{keyword}(.{{0,800}})"
    match = re.search(pattern, text, re.I)
    return match.group(0)[:600] if match else None

# ----------------------------------------------------------
# 6️⃣ PROCESS PDFs
# ----------------------------------------------------------
results = []
errors = []

pdf_files = list(PDF_FOLDER.glob("*.pdf"))
print("📂 PDFs encontrados:", len(pdf_files))

for pdf_path in tqdm(pdf_files):

    try:
        doc = fitz.open(pdf_path)
        full_text = ""
        for page in doc:
            full_text += page.get_text()
        doc.close()

        extracted_doi = extract_doi(full_text)
        matched_index = None

        # MATCH POR DOI
        if extracted_doi in doi_index:
            matched_index = doi_index[extracted_doi]

        # FALLBACK POR TÍTULO
        if matched_index is None:
            title_guess = normalize(full_text.split("\n")[0])
            if title_guess in title_index:
                matched_index = title_index[title_guess]

        if matched_index is None:
            errors.append({"Arquivo": pdf_path.name,
                           "Erro": "Nenhum match encontrado"})
            continue

        meta_row = df_meta.loc[matched_index]

        results.append({
            # Bibliometria (vem da planilha)
            "Autores": meta_row.get("Authors"),
            "Ano": meta_row.get("Year"),
            "Journal": meta_row.get("Journal"),
            "Citações": meta_row.get("Citations"),
            "País": meta_row.get("Country"),
            "Afiliação": meta_row.get("Affiliation"),

            # Metodologia
            "Tipo_Metodologia": detect_methodology(full_text),

            # Plataforma
            "Plataforma_Algoritmo": detect_platform(full_text),

            # Comportamento
            "Comportamento_Estudado": detect_behavior(full_text),

            # Achados
            "Achados_Principais": extract_section(full_text, "result"),

            # Gaps
            "Gaps_Mencionados": extract_section(full_text, "future research"),

            # Soluções
            "Solucoes_Propostas": extract_section(full_text, "implication"),

            "DOI": extracted_doi,
            "Arquivo": pdf_path.name
        })

    except Exception as e:
        errors.append({"Arquivo": pdf_path.name,
                       "Erro": str(e)})

# ----------------------------------------------------------
# 7️⃣ EXPORT
# ----------------------------------------------------------
df_final = pd.DataFrame(results)

df_final.to_csv(OUTPUT_CSV, index=False)
df_final.to_excel(OUTPUT_XLSX, index=False)

print("✅ Arquivos exportados:")
print(OUTPUT_CSV)
print(OUTPUT_XLSX)

if errors:
    pd.DataFrame(errors).to_csv(LOG_FILE, index=False)
    print("⚠️ Log de erros salvo:", LOG_FILE)

print("🎯 PROCESSAMENTO CONCLUÍDO")

📊 Total artigos na base: 2944
📄 Artigos com PDF disponível: 2930
📂 PDFs encontrados: 2508


  3%|▎         | 83/2508 [03:03<3:46:27,  5.60s/it]

MuPDF error: syntax error: could not parse color space (269 0 R)



 31%|███       | 767/2508 [10:32<05:56,  4.88it/s]

MuPDF error: syntax error: could not parse color space (632 0 R)

MuPDF error: syntax error: could not parse color space (632 0 R)

MuPDF error: syntax error: could not parse color space (837 0 R)

MuPDF error: syntax error: could not parse color space (837 0 R)

MuPDF error: syntax error: could not parse color space (1195 0 R)

MuPDF error: syntax error: could not parse color space (1195 0 R)



100%|██████████| 2508/2508 [28:43<00:00,  1.45it/s]


✅ Arquivos exportados:
/content/drive/MyDrive/Tese/ArticleSLR/artigos_analise.csv
/content/drive/MyDrive/Tese/ArticleSLR/artigos_analise.xlsx
⚠️ Log de erros salvo: /content/drive/MyDrive/Tese/ArticleSLR/log_erros.csv
🎯 PROCESSAMENTO CONCLUÍDO
